RUN THE FIRST CELL TO DEFINE REPORT CREATOR, MAKE SURE THE FILE PATH IS CORRECT, ITS AT THE BOTTOM OF THE CELL. STILL NEED TO CLEAN THIS UP, SOMETIMES STILL RUNS INTO ISSUES SO I GOTTA FIX THAT NOW THAT THERES ACTUALLY INTEREST

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from scipy.stats import gaussian_kde
from matplotlib.patches import Polygon



import numpy as np
from datetime import datetime

def get_opponent():
    opponent = input("Enter opponent")
    return opponent



# def get_pitch_heatmap(df, Pitcher, pitch_type):
#     if pitch_type == "Breaking Ball":
#         pitch_df = df.loc[(df['Pitcher'] == Pitcher) & (df['TaggedPitchType'].isin(["Slider", "Curveball"]))]
#     elif pitch_type == "Fastball":
#         pitch_df = df.loc[(df['Pitcher'] == Pitcher) & (df['TaggedPitchType'].isin(["Fastball","Four-Seam", "Sinker", "Cutter"]))]
#     elif pitch_type == "Offspeed":
#         pitch_df = df.loc[(df['Pitcher'] == Pitcher) & (df['TaggedPitchType'].isin(["ChangeUp", "Splitter"]))]
    
#      # Ensure data does not contain NaNs or infinities
#     pitch_df = pitch_df.dropna(subset=['PlateLocSide', 'PlateLocHeight'])
#     pitch_df = pitch_df[np.isfinite(pitch_df['PlateLocSide']) & np.isfinite(pitch_df['PlateLocHeight'])]
    
#     # Handle the case where pitch_df is empty after removing NaNs and infinities
#     if pitch_df.empty or len(pitch_df) < 2:
#         print(f"Not enough valid data for {Pitcher}'s {pitch_type} to compute KDE. Skipping.")
#         return None

#     x = -pitch_df['PlateLocSide']
#     y = pitch_df['PlateLocHeight']
#     k = gaussian_kde(np.vstack([x, y]))
#     xi, yi = np.mgrid[-2:2:100j, 0:5:100j]
#     zi = k(np.vstack([xi.flatten(), yi.flatten()]))

#     return xi, yi, zi.reshape(xi.shape)

def get_pitch_scatter_data(df, Pitcher, pitch_type, batter_side):
    # Filter the DataFrame for the pitcher and pitch type
    if pitch_type == "Breaking Ball":
        pitch_df = df.loc[(df['Pitcher'] == Pitcher) & 
                          (df['AutoPitchType'].isin(["Slider", "Curveball"])) & 
                          (df['BatterSide'] == batter_side)]
    elif pitch_type == "Fastball":
        pitch_df = df.loc[(df['Pitcher'] == Pitcher) & 
                          (df['AutoPitchType'].isin(["Fastball","Four-Seam", "Sinker", "Cutter"])) & 
                          (df['BatterSide'] == batter_side)]
    elif pitch_type == "Offspeed":
        pitch_df = df.loc[(df['Pitcher'] == Pitcher) & 
                          (df['AutoPitchType'].isin(["ChangeUp", "Splitter"])) & 
                          (df['BatterSide'] == batter_side)]
    
    # Clean data
    pitch_df = pitch_df.dropna(subset=['PlateLocSide', 'PlateLocHeight', 'EventRV'])
    pitch_df = pitch_df[np.isfinite(pitch_df['PlateLocSide']) & np.isfinite(pitch_df['PlateLocHeight'])]
    
    # Check if data is sufficient
    if pitch_df.empty:
        print(f"Not enough valid data for {Pitcher}'s {pitch_type} against {batter_side} batters. Skipping.")
        return None
    
 # Return the DataFrame with the required columns
    return pitch_df[['PlateLocSide', 'PlateLocHeight', 'EventRV']]

def generate_report(df, first_name, last_name):
    print(f"Generating report for {first_name} {last_name}")
    print(f"{len(df)} Pitches Thrown")
    pitcher_name = last_name + ", " + first_name
    formatted_pitcher_name = f"{first_name} {last_name}"
   # pitcher_data = df[df['Pitcher'] == pitcher_name]
    

    if df.empty:
        print(f"No data for {pitcher_name}. Skipping.")  # Diagnostic print
        return  # Skip this pitcherdef generate_report(df, first_name, last_name):
        print(f"Generating report for {first_name} {last_name}")  # Diagnostic print
    
    
    current_date = df['Date'].iloc[0]  # Format date as YYYYMMDD
    opponent = get_opponent()
    filename = f"C:/Users/TrevorWhite/Downloads/pitcher_report_{first_name}_{last_name}_{current_date}.pdf"
###MAKE SURE THE PATH IS CORRECT FOR YOU
    pitch_type_counts = pitcher_data['AutoPitchType'].value_counts()
    pitch_type_averages = pitcher_data.groupby('AutoPitchType')[['RelSpeed', 'SpinRate', 'InducedVertBreak', 'HorzBreak', 'RelHeight', 'RelSide', 'Extension']].mean()

    strikes = pitcher_data[pitcher_data['PitchCall'].isin(['StrikeCalled', 'StrikeSwinging', 'FoulBall','InPlay'])]
    strike_percentages = (strikes.groupby('AutoPitchType').size() / pitch_type_counts * 100).rename('Strike %')

    hits = len(pitcher_data[pitcher_data['PlayResult'].isin(['Single', 'Double', 'Triple', 'HomeRun'])])
    total_pitches = len(pitcher_data)
    strikeouts = len(pitcher_data[pitcher_data['KorBB'] == 'Strikeout'])
    walks = len(pitcher_data[pitcher_data['KorBB'] == 'Walk'])

    outs_pitch = (pitcher_data['Outs'].diff() > 0).sum()
    total_outs = len(pitcher_data[(pitcher_data['OutsOnPlay'] == 1) | 
                             (pitcher_data['PlayResult'].isin(['Out', 'Sacrifice'])) | 
                             (pitcher_data['KorBB'] == 'Strikeout')])




    innings = total_outs // 3  # Calculate complete innings
    remaining_outs = total_outs % 3  # Calculate the remaining outs for the fraction of the last inning

    innings_pitched = f"{innings}.{remaining_outs}"

    first_pitches = pitcher_data[pitcher_data['PitchofPA'] == 1]
    first_pitch_strikes = first_pitches[first_pitches['PitchCall'].isin(['StrikeCalled', 'StrikeSwinging', 'FoulBall','InPlay'])]
    first_pitch_strike_percentage = len(first_pitch_strikes) / len(first_pitches) * 100
    
    df = df.replace([np.inf, -np.inf], np.nan)
    #if NaN, take mean
    numeric_df = df.select_dtypes(include=[np.number])
    df[numeric_df.columns] = numeric_df.fillna(numeric_df.mean())

    

    
    df['PlateLocSide'].replace([np.inf, -np.inf], np.nan, inplace=True)
    df['PlateLocSide'].fillna(df['PlateLocSide'].median(), inplace=True)

    df['PlateLocHeight'].replace([np.inf, -np.inf], np.nan, inplace=True)
    df['PlateLocHeight'].fillna(df['PlateLocHeight'].median(), inplace=True)
#     fastball_heatmap = get_pitch_heatmap(df, pitcher_name, 'Fastball')
#     breakingball_heatmap = get_pitch_heatmap(df, pitcher_name, 'Breaking Ball')
#     offspeed_heatmap = get_pitch_heatmap(df, pitcher_name, 'Offspeed')#######
 
    
    swinging_strikes = pitcher_data[pitcher_data['PitchCall'] == 'StrikeSwinging'].groupby('AutoPitchType').size()
    total_strikes = pitcher_data[pitcher_data['PitchCall'].isin(['InPlay', 'StrikeCalled','StrikeSwinging', 'FoulBall'])].groupby('AutoPitchType').size()
    whiff_percentages = (swinging_strikes / total_strikes * 100).rename('Whiff %')
    whiff_percentages = whiff_percentages.fillna(0)
    
        # Identify in-zone pitches
    df.loc[:, 'InZone'] = df.apply(
        lambda row: -0.83 <= row['PlateLocSide'] <= 0.83 and 1.5 <= row['PlateLocHeight'] <= 3.6, axis=1)

    # Group by TaggedPitchType and count in-zone swinging strikes
    in_zone_swinging_strikes = df[(df['InZone']) & (df['PitchCall'] == 'StrikeSwinging')].groupby('AutoPitchType').size()

    # Group by TaggedPitchType and count total in-zone pitches
    in_zone_total_strikes = df[df['InZone']].groupby('AutoPitchType').size()

    # Calculate In Zone Whiff % for each pitch type and rename the series
    in_zone_whiff_percentages = (in_zone_swinging_strikes / in_zone_total_strikes * 100).rename('IZWhiff* %').fillna(0)
    


    
    pitch_type_averages = pitch_type_averages.join(strike_percentages).join(in_zone_whiff_percentages).fillna(0)
    
    # Define 'win' condition per the updated requirements
    df['win'] = ((df['PitchCall'] == 'StrikeCalled') | 
                 (df['PitchCall'] == 'StrikeSwinging') | 
                 ((df['PitchCall'] == 'FoulBall') & (df['Strikes'] < 2)) |
                 (df['OutsOnPlay'] == 1))
    
    
        # Define the Run Value (RV) for each event
    event_rv_dict = {
        'HomeRun': 1.374328827219,
        'Triple': 1.05755624961515,
        'Double': 0.766083122898271,
        'Single': 0.467292970729251,
        'FieldersChoice': -0.1955687665555,
        'Out': -0.1955687665555,
        'Sac': -0.195,
        'Error': -0.195,
        'Ball': 0.0636883289483747,
        'HitByPitch': 0.0636883289483747,
        'Foul': -0.0380502742575014,
        'StrikeCalled': -0.065092516089806,
        'StrikeSwinging': -0.118124935770601,
    }
        # Define the vertices of the plate. The vertices are based on the line plots you provided.
    plate_vertices = [(-0.83, 0.1), (0.83, 0.1), (0.65, 0.25), (0, 0.5), (-0.65, 0.25)]

    # Create a Polygon patch with these vertices
    plate = Polygon(plate_vertices, closed=True, linewidth=1, edgecolor='k', facecolor='none')

    # Calculate 'EventRV' based on 'PitchCall' and 'PlayResult'
    def calculate_event_rv(row):
        if row['PitchCall'] == 'InPlay':
            return event_rv_dict.get(row['PlayResult'], 0)  # Default to 0 if the result is not in the dictionary
        else:
            return event_rv_dict.get(row['PitchCall'], 0)  # Use PitchCall value if not 'InPlay'

    # Apply the function to each row to create the 'EventRV' column
    df['EventRV'] = df.apply(calculate_event_rv, axis=1)

    

    # Group by ball-strike count to calculate metrics
    counts = df.groupby(['Balls', 'Strikes']).agg(
        total_pitches=pd.NamedAgg(column='PitchCall', aggfunc='count'),
        wins=pd.NamedAgg(column='win', aggfunc='sum')
    )

    # Calculate win rates for each count
    # Instead of calculating win_rate, create a string representation
    counts['win_rate_str'] = counts.apply(lambda row: f"{int(row['wins'])}/{int(row['total_pitches'])}", axis=1)

    # Pivot the data to create a matrix representation with these strings
    matrix_str = counts['win_rate_str'].unstack(fill_value='0/0')
    
       # Retrieve pitch location data for each combination
    breakingball_LHH_data = get_pitch_scatter_data(df, pitcher_name, 'Breaking Ball', 'Left')
    breakingball_RHH_data = get_pitch_scatter_data(df, pitcher_name, 'Breaking Ball', 'Right')
    offspeed_LHH_data = get_pitch_scatter_data(df, pitcher_name, 'Offspeed', 'Left')
    offspeed_RHH_data = get_pitch_scatter_data(df, pitcher_name, 'Offspeed', 'Right')
    fastball_LHH_data = get_pitch_scatter_data(df, pitcher_name, 'Fastball', 'Left')
    fastball_RHH_data = get_pitch_scatter_data(df, pitcher_name, 'Fastball', 'Right')


    with PdfPages(filename) as pdf:
        fig = plt.figure(figsize=(11, 17))
        gs = GridSpec(nrows=4, ncols=3, height_ratios=[1, 2, 2, 2], wspace=0.1, hspace=0.3)
        fig.patch.set_facecolor('xkcd:light grey')

        ax1 = fig.add_subplot(gs[0, 0])
        ax1.pie(pitch_type_counts, labels=pitch_type_counts.index, autopct='%1.1f%%', startangle=90)
        ax1.set_title('Pitch Types Distribution', fontsize=16)
        ax1.set_position([0.01, 0.76, 0.35, 0.175])

        ax2 = fig.add_subplot(gs[1, 0])
        ax2.axis('off')
        basic_stats_data = [
            ['Total Pitches', total_pitches],
            ['Innings Pitched', innings_pitched],
            ['Strikeouts', strikeouts],
            ['Walks', walks],
            ['Hits', hits],
            ['First Pitch Strike %', round(first_pitch_strike_percentage, 2)]
        ]
        basic_stats_table = ax2.table(cellText=basic_stats_data, cellLoc='left', loc='center', colLabels=None)
        basic_stats_table.auto_set_font_size(False)
        basic_stats_table.set_fontsize(14)
        basic_stats_table.auto_set_column_width(col=list(range(2)))
        for (i, j), cell in basic_stats_table._cells.items():
            if i == 0:
                cell.set_text_props(weight='bold', color='white')
                cell.set_facecolor('gray')
            cell.set_height(0.2)
        ax2.set_position([0.34, 0.79, 0.35, 0.15])

        ax3 = fig.add_subplot(gs[1, 1:])
        ax3.axis('off')
        pitch_type_averages_table = ax3.table(cellText=pitch_type_averages.round(1).values,
                                            colLabels=pitch_type_averages.columns,
                                            rowLabels=pitch_type_averages.index,
                                            cellLoc='center', loc='center')
        pitch_type_averages_table.auto_set_font_size(False)
        pitch_type_averages_table.set_fontsize(13)
        pitch_type_averages_table.auto_set_column_width(col=list(range(len(pitch_type_averages.columns))))
        for (i, j), cell in pitch_type_averages_table._cells.items():
            if i == 0:
                cell.set_text_props(weight='bold', color='white')
                cell.set_facecolor('gray')
            cell.set_height(0.2)
        ax3.set_title(f"{formatted_pitcher_name} - Pitch Level Metrics", fontsize=16)
        ax3.set_position([0.45, 0.24, 0.35, 0.12])

#         ax4 = fig.add_subplot(gs[1, :])
#         strike_zone = Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none')
#         ax4.add_patch(strike_zone)
#         ax4.set_xlim(-2, 2)
#         ax4.set_ylim(0, 4.5)
#         ax4.set_aspect('equal')
#         colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'w']

#         for idx, pitch_type in enumerate(pitch_type_counts.index):
#             filtered_data = pitcher_data[pitcher_data['TaggedPitchType'] == pitch_type]
#             ax4.scatter(filtered_data['PlateLocSide'], filtered_data['PlateLocHeight'], s=80, c=colors[idx], alpha=0.5, label=pitch_type)
#         ax4.legend(title="Pitch Types", fontsize=10)
#         ax4.set_title('Pitch Locations', fontsize=20)
#         ax4.set_position([0.12, 0.43, 0.55, 0.32])
        
        
#          ax4 = fig.add_subplot(gs[2, :])
#         strike_zone = Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none')
#         ax4.add_patch(strike_zone)
#         ax4.set_xlim(-2, 2)
#         ax4.set_ylim(0, 4.5)
#         ax4.set_aspect('equal')
#         colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'w']

#         for idx, pitch_type in enumerate(pitch_type_counts.index):
#             filtered_data = pitcher_data[pitcher_data['TaggedPitchType'] == pitch_type]
#             ax4.scatter(filtered_data['PlateLocSide'], filtered_data['PlateLocHeight'], s=80, c=colors[idx], alpha=0.5, label=pitch_type)
#         ax4.legend(title="Pitch Types", fontsize=10)
#         ax4.set_title('Pitch Locations', fontsize=20)
#         ax4.set_position([0.12, 0.43, 0.55, 0.32])
        
        ax5 = fig.add_subplot(gs[0, 2:])
        ax5.axis('on')  # Change to 'on' to display the axis, which might be useful for a matrix

        # Assuming 'matrix' is the win_rate matrix you calculated earlier
        # You can use matshow or imshow here for a heatmap visualization or continue with a table if preferred

#         # For a heatmap representation:
#         cax = ax5.matshow(matrix, interpolation='nearest', cmap='coolwarm')
#         fig.colorbar(cax, ax=ax5)

        ax5.clear()  # Clear any existing content in ax5
        ax5.axis('off')  # Turn off the axis again since we're displaying a table

        # Creating the table with the matrix_str values
        cell_text = matrix_str.values.tolist()
        row_labels = matrix_str.index.tolist()
        col_labels = matrix_str.columns.tolist()

        win_rates_table = ax5.table(cellText=cell_text, cellLoc='center', rowLabels=row_labels, colLabels=col_labels, loc='center', edges='closed')
        win_rates_table.auto_set_font_size(False)
        win_rates_table.set_fontsize(14)
        win_rates_table.scale(1, 1.8)  # You might need to adjust scaling based on your layout

        # Set the title and adjust position as needed
        #
        ax5.set_position([0.71, 0.78, 0.40, 0.175])  # Adjust position and size as needed     lbwh
        ax5.set_title("Win* per Pitch Count Matrix", fontsize=14)


#         ax5 = fig.add_subplot(gs[2, 1:])
#         ax5.axis('off')
#         strike_percentages_data = [[pitch_type, round(strike_percentages[pitch_type], 2)] for pitch_type in strike_percentages.index]
#         strike_percentages_table = ax5.table(cellText=strike_percentages_data,
#                                             colLabels=['Pitch Type', 'Strike %'],
#                                             cellLoc='center', loc='center')
#         strike_percentages_table.auto_set_font_size(False)
#         strike_percentages_table.set_fontsize(14)
#         strike_percentages_table.auto_set_column_width(col=list(range(2)))
#         for (i, j), cell in strike_percentages_table._cells.items():
#             if i == 0:
#                 cell.set_text_props(weight='bold', color='white')
#                 cell.set_facecolor('gray')
#             cell.set_height(0.2)
#         ax5.set_position([0.825, 0.775, 0.35, 0.15])
        if fastball_RHH_data is not None and not fastball_RHH_data.empty:
            ax6 = fig.add_subplot(gs[1, 0])
            ax6.axis('on')  # Turn on the axis to display the scatter plot
            ax6.set_title('Fastball Locations', fontsize=16, pad = 20)

            # Generate a colormap based on EventRV values
            # Normalize EventRV values to map them to the colormap
            norm = plt.Normalize(-.2, 1.4)
            cmap = plt.cm.RdYlGn_r  # Choose a colormap that fits t

            # Color points by EventRV, applying the colormap and normalizer to the EventRV values
            scatter = ax6.scatter(fastball_RHH_data['PlateLocSide'], fastball_RHH_data['PlateLocHeight'], 
                                  c=fastball_RHH_data['EventRV'],  # This needs to be the EventRV values for the pitches
                                  s=80, cmap=cmap, norm=norm, alpha=0.5, label='Fastball (RHH)')

#             # Add a colorbar to show the EventRV scale
#             cbar = plt.colorbar(scatter, ax=ax6)
#             cbar.set_label('Event Run Value')

            # Add a rectangle to indicate the strike zone
            ax6.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
            ax6.add_patch(plate)
            ax6.set_xlim(-2, 2)
            ax6.set_ylim(0, 4.5)
            ax6.set_aspect('equal')
            ax6.set_position([0.26, 0.575, 0.3, 0.15])  # Adjust as needed
            
                    # Fastball LHH
        if fastball_LHH_data is not None and not fastball_LHH_data.empty:
            ax7 = fig.add_subplot(gs[2, 0])
            ax7.axis('on')
            #ax7.set_title('Fastball Locations (LHH)', fontsize=16)
            norm = plt.Normalize(-.2, 1.4)
            cmap = plt.cm.RdYlGn_r
            scatter = ax7.scatter(fastball_LHH_data['PlateLocSide'], fastball_LHH_data['PlateLocHeight'],
                                  c=fastball_LHH_data['EventRV'], s=80, cmap=cmap, norm=norm, alpha=0.5, label='Fastball (LHH)')
            ax7.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
            ax7.add_patch(Polygon(plate_vertices, closed=True, linewidth=1, edgecolor='k', facecolor='none'))
            ax7.set_xlim(-2, 2)
            ax7.set_ylim(0, 4.5)
            ax7.set_aspect('equal')
            ax7.set_position([0.26, 0.4, 0.3, 0.15])
            


        # Repeat similar steps for Breaking Ball and Offspeed, adjusting ax references and data sources accordingly.
        # Breaking Ball RHH
        if breakingball_RHH_data is not None and not breakingball_RHH_data.empty:
            ax8 = fig.add_subplot(gs[1, 1])
            ax8.axis('on')
            ax8.set_title('Breaking Ball Locations', fontsize=16, pad = 20)
            norm = plt.Normalize(-.2, 1.4)
            cmap = plt.cm.RdYlGn_r
            scatter = ax8.scatter(breakingball_RHH_data['PlateLocSide'], breakingball_RHH_data['PlateLocHeight'],
                                  c=breakingball_RHH_data['EventRV'], s=80, cmap=cmap, norm=norm, alpha=0.5, label='Breaking Ball (RHH)')
            ax8.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
            ax8.add_patch(Polygon(plate_vertices, closed=True, linewidth=1, edgecolor='k', facecolor='none'))
            ax8.set_xlim(-2, 2)
            ax8.set_ylim(0, 4.5)
            ax8.set_aspect('equal')
            ax8.set_position([0.56, 0.575, 0.3, 0.15])


        # Breaking Ball LHH
        if breakingball_LHH_data is not None and not breakingball_LHH_data.empty:
            ax9 = fig.add_subplot(gs[2, 1])
            ax9.axis('on')
           # ax9.set_title('Breaking Ball Locations (LHH)', fontsize=16)''
            norm = plt.Normalize(-.2, 1.4)
            cmap = plt.cm.RdYlGn_r
            scatter = ax9.scatter(breakingball_LHH_data['PlateLocSide'], breakingball_LHH_data['PlateLocHeight'],
                                  c=breakingball_LHH_data['EventRV'], s=80, cmap=cmap, norm=norm, alpha=0.5, label='Breaking Ball (LHH)')
            ax9.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
            ax9.add_patch(Polygon(plate_vertices, closed=True, linewidth=1, edgecolor='k', facecolor='none'))
            ax9.set_xlim(-2, 2)
            ax9.set_ylim(0, 4.5)
            ax9.set_aspect('equal')
            ax9.set_position([0.56, 0.4, 0.3, 0.15])


        # Offspeed RHH
        if offspeed_RHH_data is not None and not offspeed_RHH_data.empty:
            ax10 = fig.add_subplot(gs[1, 2])
            ax10.axis('on')
            ax10.set_title('Offspeed Locations', fontsize=16, pad = 20)
            norm = plt.Normalize(-.2, 1.4)
            cmap = plt.cm.RdYlGn_r
            scatter = ax10.scatter(offspeed_RHH_data['PlateLocSide'], offspeed_RHH_data['PlateLocHeight'],
                                   c=offspeed_RHH_data['EventRV'], s=80, cmap=cmap, norm=norm, alpha=0.5, label='Offspeed (RHH)')
            ax10.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
            ax10.add_patch(Polygon(plate_vertices, closed=True, linewidth=1, edgecolor='k', facecolor='none'))
            ax10.set_xlim(-2, 2)
            ax10.set_ylim(0, 4.5)
            ax10.set_aspect('equal')
            ax10.set_position([0.86, 0.575, 0.3, 0.15])


        # Offspeed LHH
        if offspeed_LHH_data is not None and not offspeed_LHH_data.empty:
            ax11 = fig.add_subplot(gs[2, 2])
            ax11.axis('on')
           # ax11.set_title('Offspeed Locations (LHH)', fontsize=16)
            cmap = plt.cm.RdYlGn_r
            norm = plt.Normalize(-.2, 1.4)
            scatter = ax11.scatter(offspeed_LHH_data['PlateLocSide'], offspeed_LHH_data['PlateLocHeight'],
                                   c=offspeed_LHH_data['EventRV'], s=80, cmap=cmap, norm=norm, alpha=0.5, label='Offspeed (LHH)')
            ax11.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
            ax11.add_patch(Polygon(plate_vertices, closed=True, linewidth=1, edgecolor='k', facecolor='none'))
            ax11.set_xlim(-2, 2)
            ax11.set_ylim(0, 4.5)
            ax11.set_aspect('equal')
            
            ax11.set_position([0.86, 0.4, 0.3, 0.15])
            # Define a new subplot for the colorbar
            # Define a new subplot for the colorbar, initially invisible
        ax12 = fig.add_subplot(gs[1, -1:])
        ax12.axis('off')

        # Create a colorbar in the specified axes
        cbar = plt.colorbar(scatter, cax=ax12, orientation='vertical')

        # Adjust colorbar position and width relative to its parent axes (ax12)
        # The values 0.05, 0.1, 0.15, and 0.8 are example proportions; adjust as needed
        cbar.ax.set_position([.002, 0.4, 0.03, 0.3])  # This positions the colorbar on the right side within ax12

        # Label the max and min of the colorbar
        ax12.text(.035, 0.4, '-0.2 - Out', va='bottom', ha='left', transform=fig.transFigure, fontsize=10)
        ax12.text(.035, 0.7, '1.4 - Home Run', va='top', ha='left', transform=fig.transFigure, fontsize=10)

        
                # Creating a new axes for the label right next to the colorbar
        ax13 = fig.add_subplot(gs[1, -1])
        ax13.axis('off')  # Turn off the axis
        ax13.set_position([0.02, 0.4, 0.05, 0.3])  # Adjust position next to the colorbar

        # Adding the label as text within ax13
        ax13.text(0.5, 0.5, 'Standardized* Event Run Value', rotation=270, fontsize = 12, verticalalignment='center', horizontalalignment='center', transform=ax13.transAxes)
        
        # Assuming the figure (fig) is already defined and set up

        # Define a new subplot for the text boxes, making it invisible
        ax14 = fig.add_subplot(gs[1, -1:])
        ax14.axis('off')  # Turn off the axes

        # Add text boxes within ax14, positioned at the specified coordinates and italicized
        ax14.text(0.175, 0.65, 'vs. RHH', transform=fig.transFigure, fontsize=24, style='italic', ha='center', va='center')
        ax14.text(0.175, 0.475, 'vs. LHH', transform=fig.transFigure, fontsize=24, style='italic', ha='center', va='center')
        # Define a new subplot for the text boxes at the bottom, making it invisible
#         ax15 = fig.add_subplot(gs[2:, 0:])
#         ax15.axis('off')  # Turn off the axes

#         # Add placeholder text boxes within ax15, positioned across the bottom
#         ax15.text(0.01, 0.1, '*Win is defined as a strike or out', transform=ax15.transAxes, fontsize=10, ha='center', va='bottom')
#         ax15.text(0.5, 0.1, '*Standardized Run Values represent average runs added because of an outcome, irrespective of actual game situation', transform=ax15.transAxes, fontsize=10, ha='center', va='center')
#         ax15.text(0.75, 0.1, '*IZWhiff = Whiff % on pitches in the zone', transform=ax15.transAxes, fontsize=10, ha='center', va='bottom')

#         if fastball_heatmap:
#             ax6 = fig.add_subplot(gs[2, 0])
#             ax6.axis('off')
#             ax6.set_title('Fastball Heatmap', fontsize=16)
#             ax6.imshow(fastball_heatmap[2], cmap='Oranges', interpolation='nearest', extent=[-2, 2, 0, 5], origin='lower')
#             ax6.set_aspect('equal')
#             ax6.set_position([0.575, 0.6, 0.35, 0.15])
#             #add a rectangle to the plot of the heatmap to show the strike zone
#             ax6.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
#             #flip the x-axis so that the plot is oriented the same way as the strike zone
#             ax6.invert_yaxis()
#             #shidt the heatmap 90 degrees so the it is turned 90 degrees clockwise
#             ax6.imshow(np.rot90(fastball_heatmap[2],3), cmap='Oranges', interpolation='nearest', extent=[-2, 2, 0, 5], origin='lower')

        
        
        
#         if breakingball_heatmap:
#             ax7 = fig.add_subplot(gs[2, 1])
#             ax7.axis('off')
#             ax7.set_title('Breaking Ball Heatmap', fontsize=16)
#             ax7.imshow(breakingball_heatmap[2], cmap='Oranges', interpolation='nearest', extent=[-2, 2, 0, 5], origin='lower')
#             ax7.set_aspect('equal')
#             ax7.set_position([0.855, 0.6, 0.35, 0.15])
#             #add a rectangle to the plot of the heatmap to show the strike zone
#             ax7.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
#             #flip the x-axis so that the plot is oriented the same way as the strike zone
#             ax7.invert_yaxis()
#             #shidt the heatmap 90 degrees so the it is turned 90 degrees clockwise
#             ax7.imshow(np.rot90(breakingball_heatmap[2],3), cmap='Oranges', interpolation='nearest', extent=[-2, 2, 0, 5], origin='lower')


    
#         if offspeed_heatmap:
#             ax8 = fig.add_subplot(gs[3, 0])
#             ax8.axis('off')
#             ax8.set_title('Offspeed Heatmap', fontsize=16)
#             ax8.imshow(offspeed_heatmap[2], cmap='Oranges', interpolation='nearest', extent=[-2, 2, 0, 5], origin='lower')
#             ax8.set_aspect('equal')
#             ax8.set_position([0.85, 0.42, 0.35, 0.15])
#             #add a rectangle to the plot of the heatmap to show the strike zone
#             ax8.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
#             #flip the x-axis so that the plot is oriented the same way as the strike zone
#             ax8.invert_yaxis()
#             #shidt the heatmap 270 degrees so the it is turned 90 degrees clockwise
#             ax8.imshow(np.rot90(offspeed_heatmap[2], 3), cmap='Oranges', interpolation='nearest', extent=[-2, 2, 0, 5], origin='lower')
#             #ax8.imshow(np.rot90(offspeed_heatmap[2], 1), cmap='Oranges', interpolation='nearest', extent=[-2, 2, 0, 5], origin='lower')

#         ax9 = fig.add_subplot(gs[3, 1:])  # Modify as needed for your layout.
#         ax9.axis('off')
#         whiff_percentages_data = [[pitch_type, round(whiff_percentages[pitch_type], 2)] for pitch_type in whiff_percentages.index]
#         whiff_percentages_table = ax9.table(cellText=whiff_percentages_data,
#                                             colLabels=['Pitch Type', 'Whiff %'],
#                                             cellLoc='center', loc='center')
#         whiff_percentages_table.auto_set_font_size(False)
#         whiff_percentages_table.set_fontsize(14)
#         whiff_percentages_table.auto_set_column_width(col=list(range(2)))
#         for (i, j), cell in whiff_percentages_table._cells.items():
#             if i == 0:
#                 cell.set_text_props(weight='bold', color='white')
#                 cell.set_facecolor('gray')
#             cell.set_height(0.2)
#         ax9.set_position([0.59, 0.425, 0.35, 0.15])
        
        #fig.tight_layout()
        
        # Define the footer text using plt.figtext at the bottom of the figure
        plt.figtext(0.95, 0.99999999, '*Win is defined as a strike or out', 
                    fontsize=10, ha='left', va='top')

        plt.figtext(0.01, 0.23, '*Standardized Run Values represent average runs added because of an outcome, irrespective of actual game situation', 
                    fontsize=10, ha='left', va='bottom')

        plt.figtext(0.99, 0.23, '*IZWhiff = Whiff % on pitches in the zone', 
                    fontsize=10, ha='center', va='bottom')
        plt.suptitle(f"{formatted_pitcher_name} - {current_date} vs. {opponent}", fontsize=28, x=0, y=1, ha='left', va='top')
        pdf.savefig(fig, bbox_inches='tight')
        #pdf.close()

df = pd.read_csv(r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\20241010-UofSanDiego-Private-2_unverified.csv", low_memory=False)


# Filter DataFrame for team 'San_Tor'
san_tor_df = df[df['PitcherTeam'] == 'SAN_TOR']

# Accessing the first value of the 'Pitcher' column, splitting it by comma, and stripping whitespace
#last_name, first_name = [name.strip() for name in pitcher_data['Pitcher'].iloc[0].split(",")]


# Now you have two objects, first_name and last_name
#print("First Name:", first_name)
#print("Last Name:", last_name)

#pitcher_name = get_pitcher_name()
#generate_report(df, pitcher_name)



In [ ]:
import pybaseball
print(dir(pybaseball))

RUN THE CELL BELOW TO RUN THE REPORT FOR EVERY SD PITCHER IN THE DATA

In [ ]:
# CODE TO TEST FIRST INSTANCE pitcher = san_tor_df['Pitcher'].iloc[0]   
# Iterate over each unique pitcher    
for pitcher in san_tor_df['Pitcher'].unique():
   
    print(f"Processing: {pitcher}")  # Diagnostic print
    last_name, first_name = pitcher.split(', ')
    pitcher_data = df[df['Pitcher'] == pitcher]
    if pitcher_data.empty:
        print(f"No data, Skipping.")  # Diagnostic print
        continue  # Skip this pitcherdef generate_report(df, first_name, last_name):
    cmap = plt.cm.RdYlGn_r
    generate_report(pitcher_data, first_name.strip(), last_name.strip())
    print(f"Finished processing: {pitcher}")  # Diagnostic print

RUN THE CELL BELOW TO RUN IT FOR A SINGLE PITCHER, USE THE # IN PITCHER_NAMES[1] TO DEFINE WHICH PITCHER YOU WANT, STARTS AT 0 SO THE SP WILL BE 0 AND SO ON.

In [ ]:
pitcher_names = san_tor_df['Pitcher'].unique().tolist()


pitcher = pitcher_names[1] 
print(f"Processing: {pitcher}")  # Diagnostic print
last_name, first_name = pitcher.split(', ')
pitcher_data = df[df['Pitcher'] == pitcher]
generate_report(pitcher_data, first_name.strip(), last_name.strip())

RUN VALUES I FOUND FROM SOME GUY, DOES THE JOB TO SCALE OUTCOMES NUMERICALLY

In [ ]:
#Set Run Value for each Description
home_run = 1.374328827219,
triple = 1.05755624961515,
double = 0.766083122898271,
single = 0.467292970729251
ball = 0.0636883289483747,
hit_by_pitch = 0.0636883289483747,
foul = -0.0380502742575014,
called_strike = -0.065092516089806,
swinging_strike = -0.118124935770601,
fielders_choice = -0.1955687665555,
field_out = -0.1955687665555,
Sac_fly = -0.236889645519856,
field_error = -0.236889645519856

In [ ]:
# import requests

# def get_report_date():
#     reportdate = input("Enter the date of report - ex: 00/00/0000 ")
#     return reportdate

# def get_primary_sheet_data():
#     primary_sheet_url = 'https://docs.google.com/spreadsheets/d/1f3s-cBGXXdPTS0DJMH58CWk0GeOzUL8DgN93aYzFXHo/export?format=csv'
#     response = requests.get(primary_sheet_url)
#     if response.status_code == 200:
#         return pd.read_csv(pd.compat.StringIO(response.text))
#     else:
#         raise Exception("Failed to access the primary Google Sheet")

# def find_trackman_url(df, report_date):
#     # Assuming the date is in a column named 'Date' and the format matches exactly
#     match = df[df['Date'] == report_date]
#     if not match.empty:
#         # Assuming the URL is in the 'Trackman' column
#         return match['Trackman'].iloc[0]
#     else:
#         return None
    

# def get_trackman_data(url):
#     export_url = url.replace('/edit?usp=sharing', '/export?format=csv')
#     response = requests.get(export_url)
#     if response.status_code == 200:
#         return pd.read_csv(pd.compat.StringIO(response.text))
#     else:
#         raise Exception("Failed to access the Trackman Google Sheet")

# def main():
#     report_date = get_report_date()
#     primary_df = get_primary_sheet_data()
#     trackman_url = find_trackman_url(primary_df, report_date)
    
#     if trackman_url:
#         trackman_df = get_trackman_data(trackman_url)
#         print(trackman_df)
#     else:
#         print("No Trackman data available for the given date.")

# main()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the CSV file into a pandas DataFrame
file_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)

# Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

# Mapping for glove/arm side based on PitcherThrows
def get_glove_or_arm_side(row):
    if row['Pitcherthrows'] == 'Right':
        if row['Horzbreak'] > 0:
            return 'Arm Side'
        else:
            return 'Glove Side'
    elif row['Pitcherthrows'] == 'Left':
        if row['Horzbreak'] > 0:
            return 'Glove Side'
        else:
            return 'Arm Side'
    return None

# Apply the function to create a new column in the dataframe
df['Side'] = df.apply(get_glove_or_arm_side, axis=1)

# Set up the color palette based on pitch type
pitch_types = df['Autopitchtype'].unique()
palette = sns.color_palette('hsv', len(pitch_types))
color_map = dict(zip(pitch_types, palette))

# Loop through each pitcher and create a scatterplot for each
for pitcher in df['Pitcher'].unique():
    plt.figure(figsize=(10, 6))
    pitcher_data = df[df['Pitcher'] == pitcher]
    
    # Plot Arm Side pitches
    arm_side_data = pitcher_data[pitcher_data['Side'] == 'Arm Side']
    sns.scatterplot(
        data=arm_side_data,
        x='Horzbreak',
        y='Inducedvertbreak',
        hue='Autopitchtype',
        palette=color_map,
        s=100,  # Size of the points
        edgecolor='black',
        marker='o',  # Marker for arm side
        legend='full'  # To make sure it only shows the pitch types
    )
    
    # Plot Glove Side pitches
    glove_side_data = pitcher_data[pitcher_data['Side'] == 'Glove Side']
    sns.scatterplot(
        data=glove_side_data,
        x='Horzbreak',
        y='Inducedvertbreak',
        hue='Autopitchtype',
        palette=color_map,
        s=100,  # Size of the points
        edgecolor='black',
       
        legend=False  # No extra legend entry for glove side
    )

    # Set the axis limits to match the image (-25 to 25)
    plt.xlim(-25, 25)
    plt.ylim(-25, 25)

    # Labeling and setting up axes with ticks at 10, 20 intervals
    plt.xticks([-20, -10, 0, 10, 20])
    plt.yticks([-20, -10, 0, 10, 20])


   # Adding the "Arm Side" and "Glove Side" labels in the corners with a box around the text
    plt.text(20, -23, 'Arm Side', fontsize=12, verticalalignment='center', horizontalalignment='center',
             bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))
    plt.text(-20, -23, 'Glove Side', fontsize=12, verticalalignment='center', horizontalalignment='center',
             bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))


    # Draw the center lines to divide the plot into four quadrants
    plt.axvline(0, color='grey', linestyle='--')
    plt.axhline(0, color='grey', linestyle='--')

    # Adding a title
    plt.title(f'Pitch Breaks: Horizontal vs Vertical Break for Pitcher {pitcher}')
    plt.xlabel('Horizontal Break (in)')
    plt.ylabel('Induced Vertical Break (in)')

    # Customize legend (position outside plot)
    plt.legend(title='Auto* Pitch Type', bbox_to_anchor=(1.05, 1), loc='upper left')

    # Show the plot for this pitcher
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load your CSV file (adjust path as needed)
file_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)

# Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

# Replace ':' with '.' in Tilt for calculation
def replace_colon_with_dot(tilt_value):
    try:
        # Ensure that Tilt is a string, and replace the colon with a dot
        return tilt_value.replace(':', '.') if isinstance(tilt_value, str) else np.nan
    except AttributeError:
        return np.nan  # Handle missing or invalid values

# Revert '.' back to ':' after calculation
def replace_dot_with_colon(decimal_tilt):
    if isinstance(decimal_tilt, str):
        return decimal_tilt.replace('.', ':')
    try:
        tilt_str = str(decimal_tilt)
        return tilt_str.replace('.', ':')
    except:
        return np.nan

# Apply the conversion of Tilt to dot format for median calculation
if 'Tilt' in df.columns:
    df['Tilt_dot'] = df['Tilt'].apply(replace_colon_with_dot)

# Identify in-zone pitches
df.loc[:, 'Inzone'] = df.apply(
    lambda row: -0.83 <= row['Platelocside'] <= 0.83 and 1.5 <= row['Platelocheight'] <= 3.6, axis=1)

# Group data by AutoPitchType and calculate relevant stats
def calculate_pitch_metrics(pitcher_data):
    # Count pitches for each type
    pitch_type_counts = pitcher_data['Autopitchtype'].value_counts().rename('Count')
    
    # Calculate averages for each pitch type (excluding Tilt)
    avg_cols = ['Relspeed', 'Spinrate', 'Inducedvertbreak', 'Horzbreak', 
                'Relheight', 'Relside', 'Extension', 'Vertapprangle', 'Horzapprangle']
    
    pitch_type_averages = pitcher_data.groupby('Autopitchtype')[avg_cols].mean()

    # Calculate the median Tilt (with '.' instead of ':') for each pitch type
    pitch_type_median_tilt = pitcher_data.groupby('Autopitchtype')['Tilt_dot'].median().rename('Tilt')

    # Calculate strike percentages
    strikes = pitcher_data[pitcher_data['Pitchcall'].isin(['StrikeCalled', 'StrikeSwinging', 'FoulBall', 'InPlay'])]
    strike_percentages = (strikes.groupby('Autopitchtype').size() / pitch_type_counts * 100).rename('Strike %')

    # Calculate whiff percentages
    swinging_strikes = pitcher_data[pitcher_data['Pitchcall'] == 'StrikeSwinging'].groupby('Autopitchtype').size()
    total_strikes = pitcher_data[pitcher_data['Pitchcall'].isin(['InPlay', 'StrikeCalled', 'StrikeSwinging', 'FoulBall'])].groupby('Autopitchtype').size()
    whiff_percentages = (swinging_strikes / total_strikes * 100).rename('Whiff %')

    # Calculate max velocity
    max_velocity = pitcher_data.groupby('Autopitchtype')['Relspeed'].max().rename('Max velo')

    # Calculate InZone %
    in_zone_total_strikes = pitcher_data[pitcher_data['Inzone']].groupby('Autopitchtype').size()
    in_zone_percentage = (in_zone_total_strikes / pitch_type_counts * 100).rename('InZone %').fillna(0)

    # Calculate InZone Whiff %
    in_zone_swinging_strikes = pitcher_data[(pitcher_data['Inzone']) & (pitcher_data['Pitchcall'] == 'StrikeSwinging')].groupby('Autopitchtype').size()
    in_zone_whiff_percentages = (in_zone_swinging_strikes / in_zone_total_strikes * 100).rename('InZone Whiff %').fillna(0)

    # Calculate Chase %
    out_of_zone_swings = pitcher_data[(~pitcher_data['Inzone']) & (pitcher_data['Pitchcall'].isin(['StrikeSwinging', 'FoulBall', 'InPlay']))].groupby('Autopitchtype').size()
    total_out_of_zone_pitches = pitcher_data[~pitcher_data['Inzone']].groupby('Autopitchtype').size()
    chase_percentage = (out_of_zone_swings / total_out_of_zone_pitches * 100).rename('Chase %').fillna(0)

    # Handle NaN and Inf values
    pitch_type_averages = pitch_type_averages.replace([np.inf, -np.inf], np.nan)
    pitch_type_averages = pitch_type_averages.fillna(pitch_type_averages.mean())

    # Join calculated stats to the main table and add the Count of pitches, max velocity, median Tilt, and other percentages
    pitch_type_averages = (pitch_type_counts.to_frame().join(pitch_type_averages)
                           .join(max_velocity)
                           .join(pitch_type_median_tilt)
                           .join(strike_percentages)
                           .join(whiff_percentages)
                           .join(in_zone_percentage)
                           .join(in_zone_whiff_percentages)
                           .join(chase_percentage)
                           .fillna(0))

    # Revert the median Tilt back to ':' format for presentation
    pitch_type_averages['Tilt'] = pitch_type_averages['Tilt'].apply(replace_dot_with_colon)

    # Ensure the correct column order
    correct_order = ['Count', 'Relspeed', 'Max velo', 'Spinrate', 'Inducedvertbreak', 'Horzbreak', 
                     'Relheight', 'Relside', 'Extension', 'Tilt', 'Vertapprangle', 
                     'Horzapprangle', 'Strike %', 'Whiff %', 'InZone %', 'InZone Whiff %', 'Chase %']
    
    pitch_type_averages = pitch_type_averages[correct_order]

    return pitch_type_averages

# Loop through each pitcher and create the table visualization
for pitcher in df['Pitcher'].unique():
    pitcher_data = df[df['Pitcher'] == pitcher]
    pitch_type_averages = calculate_pitch_metrics(pitcher_data)

    # Rename columns explicitly to match the final display order
    expected_columns = ['Count', 'Vel', 'Max Velo', 'Spin', 'iVB', 'HB', 'yRel', 'xRel', 'Ext', 'Tilt', 'VAA', 'HAA', 'Strike%', 'Whiff%', 'Zone%', 'zWhiff%', 'Chase%']
    pitch_type_averages.columns = expected_columns

    # Create figure for visualization
    fig = plt.figure(figsize=(14, 8))
    ax = fig.add_subplot(111)
    ax.axis('off')  # Turn off the axis
    
    # Create the table
    table = ax.table(cellText=pitch_type_averages.round(1).values,
                     colLabels=pitch_type_averages.columns,
                     rowLabels=pitch_type_averages.index,
                     cellLoc='center', loc='center')

    # Adjust table font size, column widths, and adjust cell height for padding
    table.auto_set_font_size(False)
    table.set_fontsize(13)
    table.auto_set_column_width(col=list(range(len(pitch_type_averages.columns))))
    
    # Adjust cell properties for padding by modifying width and height
    for (i, j), cell in table._cells.items():
        if i == 0:  # Header row
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('gray')
        cell.set_height(0.15)  # Increase row height for padding
        cell.set_width(0.25)   # Adjust column width for padding
    
    # Set title and adjust position
    ax.set_title(f"{pitcher} - Pitch Level Metrics", fontsize=18)
    
    # Display the plot for this pitcher
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Function to convert Tilt from HH:MM format to float (e.g., 1:45 -> 1.75)
def convert_tilt_to_float(tilt_value):
    try:
        if isinstance(tilt_value, str) and ':' in tilt_value:
            hours, minutes = tilt_value.split(':')
            return float(hours) + float(minutes) / 60
        return np.nan
    except:
        return np.nan

# Load your CSV file (adjust path as needed)
file_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)

# Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

# Convert the Tilt column to float
df['Tilt_float'] = df['Tilt'].apply(convert_tilt_to_float)

# Set up a color palette for pitch types
pitch_types = df['Autopitchtype'].unique()
palette = sns.color_palette('hsv', len(pitch_types))
color_map = dict(zip(pitch_types, palette))

# Function to create polar plots for each pitcher
def create_polar_plots(pitcher_data, pitcher_name):
    # Filter out rows where Tilt_float is NaN
    pitcher_data = pitcher_data.dropna(subset=['Tilt_float'])

    # Convert Tilt to radians (each hour on the clock represents 30 degrees)
    tilt_radians = np.radians(pitcher_data['Tilt_float'] * 30)

    # Create the figure with two subplots (polar plots)
    fig, axs = plt.subplots(1, 2, subplot_kw={'projection': 'polar'}, figsize=(12, 6))

    # Set 12 o'clock at the top of the plot (N for north) and ensure clockwise direction
    axs[0].set_theta_zero_location('N')
    axs[0].set_theta_direction(-1)
    axs[1].set_theta_zero_location('N')
    axs[1].set_theta_direction(-1)

    # Plot 1: Velocity (Relspeed) vs. Tilt, color by pitch type
    sc1 = axs[0].scatter(tilt_radians, pitcher_data['Relspeed'], c=pitcher_data['Autopitchtype'].map(color_map), s=100)
    axs[0].set_title('Velocity vs. Tilt', fontsize=14)
    axs[0].set_xlabel('Tilt (Clock)')
    axs[0].set_ylabel('Velocity')
    axs[0].set_ylim(70, 95)  # Set y-axis limits for Velocity
    axs[0].set_xticks(np.radians(np.arange(0, 360, 30)))  # Positions for clock ticks
    axs[0].set_xticklabels(['12', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11'])

    # Plot 2: Spin Rate (Spinrate) vs. Tilt, color by pitch type
    sc2 = axs[1].scatter(tilt_radians, pitcher_data['Spinrate'], c=pitcher_data['Autopitchtype'].map(color_map), s=100)
    axs[1].set_title('Spin Rate vs. Tilt', fontsize=14)
    axs[1].set_xlabel('Tilt (Clock)')
    axs[1].set_ylabel('Spin Rate')
    axs[1].set_ylim(1500, 2700)  # Set y-axis limits for Spin Rate
    axs[1].set_xticks(np.radians(np.arange(0, 360, 30)))  # Positions for clock ticks
    axs[1].set_xticklabels(['12', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11'])

    # Adjust titles, labels, and layout
    plt.suptitle(f'Pitcher: {pitcher_name}', fontsize=16)

    # Add legend for pitch types
    handles = [plt.Line2D([0], [0], marker='o', color=palette[i], markersize=10, linestyle='') for i in range(len(pitch_types))]
    plt.legend(handles, pitch_types, title='Auto Pitch Type', bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# Loop through each pitcher and create the polar plots
for pitcher in df['Pitcher'].unique():
    pitcher_data = df[df['Pitcher'] == pitcher]
    create_polar_plots(pitcher_data, pitcher)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load your CSV file (adjust path as needed)
file_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)

# Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

# Set up a color palette for pitch types
pitch_types = df['Autopitchtype'].unique()
palette = sns.color_palette('hsv', len(pitch_types))
color_map = dict(zip(pitch_types, palette))

# Function to create the release point plot for each pitcher
def create_release_plot(pitcher_data, pitcher_name):
    # Create the figure
    plt.figure(figsize=(8, 6))

    # Scatter plot of RelSide (x-axis) vs. RelHeight (y-axis) colored by pitch type
    sns.scatterplot(
        data=pitcher_data,
        x='Relside',
        y='Relheight',
        hue='Autopitchtype',
        palette=color_map,
        s=100,  # Size of the points
        edgecolor='black'  # Add a black border to the points for clarity
    )

    # Set the axis limits
    plt.xlim(-4, 4)  # Horizontal release point range
    plt.ylim(2, 7)   # Vertical release point range

    # Add labels and title
    plt.xlabel('Horizontal Release Point')
    plt.ylabel('Vertical Release Point')
    plt.title(f'Release Point for {pitcher_name}', fontsize=16)

    # Customize the legend
    plt.legend(title='Pitch Type', bbox_to_anchor=(1.05, 1), loc='upper left')

    # Show the plot
    plt.tight_layout()
    plt.show()

# Loop through each pitcher and create the release plot
for pitcher in df['Pitcher'].unique():
    pitcher_data = df[df['Pitcher'] == pitcher]
    create_release_plot(pitcher_data, pitcher)


In [ ]:
os.environ["GITHUB_TOKEN"]  # was a hardcoded PAT, revoked =  GITHUB_TOKEN

In [ ]:
import os
import shutil
import pandas as pd
import re

# Define the path to search for files
directory = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs"
processed_folder = os.path.join(directory, 'processed')
master_file_path = os.path.join(directory, 'usd_baseball_TM_master_file.csv')

# Create the processed folder if it doesn't exist
if not os.path.exists(processed_folder):
    os.makedirs(processed_folder)

# Regex pattern to match the desired CSV filenames
file_pattern = r"\d{8}-UofSanDiego-(Private|BP)-\d+_unverified\.csv"

# Gather new dataframes
csv_dataframes = []

# Loop through and collect matching files
for filename in os.listdir(directory):
    if re.match(file_pattern, filename):
        file_path = os.path.join(directory, filename)
        try:
            df = pd.read_csv(file_path)
            csv_dataframes.append(df)
            print(f"Appended data from {filename}")
        except Exception as e:
            print(f"Error reading {filename}: {e}")
            continue
        # Move to processed regardless
        try:
            shutil.move(file_path, os.path.join(processed_folder, filename))
            print(f"Moved {filename} to {processed_folder}")
        except Exception as e:
            print(f"Error moving {filename}: {e}")

# If we found any new files, append them to the master file (local), else do nothing
if csv_dataframes:
    new_combined_df = pd.concat(csv_dataframes, ignore_index=True)
    if os.path.exists(master_file_path):
        try:
            master_df = pd.read_csv(master_file_path)
            updated_df = pd.concat([master_df, new_combined_df], ignore_index=True)
            print(f"Appended new data to existing master file at {master_file_path}")
        except Exception as e:
            print(f"Error reading existing master file: {e}")
            updated_df = new_combined_df
    else:
        updated_df = new_combined_df
        print(f"No existing master file found. Creating new one at {master_file_path}")

    try:
        updated_df.to_csv(master_file_path, index=False)
        print(f"Saved updated master file to {master_file_path}")
    except Exception as e:
        print(f"Error saving updated master file: {e}")
else:
    print("No new CSV files matched the pattern.")


In [ ]:
updated_df.to_csv(master_file_path, index=False)

In [ ]:
import pandas as pd

# Path to local OneDrive master file
local_master_file_path2 = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv"

# Use the online master CSV (Trackman file) as the base for appending
TRACKMAN_URL = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/usd_baseball_TM_master_file.csv"

try:
    # Try to read online master file first
    online_master_df = pd.read_csv(TRACKMAN_URL)
    print(f"Loaded Trackman master file from {TRACKMAN_URL}")

    # Append new_combined_df to the online master dataframe
    updated_master_df2 = pd.concat([online_master_df, new_combined_df], ignore_index=True)
    print("Appended new combined dataframe to Trackman master file from GitHub.")
except Exception as e:
    print(f"Could not load online Trackman master file: {e}")
    # If read fails, just use new_combined_df
    updated_master_df2 = new_combined_df.copy()
    print("Using only new combined dataframe for master file.")

# Save result to OneDrive file location
try:
    updated_master_df2.to_csv(local_master_file_path2, index=False)
    print(f"Appended new data to local master file at {local_master_file_path2}.")
except Exception as e:
    print(f"Error saving updated master file at {local_master_file_path2}: {e}")



In [ ]:
import pandas as pd

# Set the URL
TRACKMAN_URL = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/usd_baseball_TM_master_file.csv"

# Load the CSV into a DataFrame
try:
    df_trackman = pd.read_csv(TRACKMAN_URL)
    print("Successfully loaded Trackman master file.")

    # Handle 'Date' by explicitly specifying date format parsing if possible
    if 'Date' in df_trackman.columns:
        # Try both common formats: %Y-%m-%d and %m/%d/%Y; fallback to automatic/robust parse
        # First try ISO (2025-11-01)
        df_trackman['Date_parsed'] = pd.to_datetime(df_trackman['Date'], format='%Y-%m-%d', errors='coerce')
        # Where that failed (NaT), try M/D/YYYY etc.
        mask_not_parsed = df_trackman['Date_parsed'].isna()
        if mask_not_parsed.any():
            df_trackman.loc[mask_not_parsed, 'Date_parsed'] = pd.to_datetime(
                df_trackman.loc[mask_not_parsed, 'Date'], format='%m/%d/%Y', errors='coerce'
            )
        # If still missing, let pandas try to parse the rest
        mask_still_not_parsed = df_trackman['Date_parsed'].isna()
        if mask_still_not_parsed.any():
            df_trackman.loc[mask_still_not_parsed, 'Date_parsed'] = pd.to_datetime(
                df_trackman.loc[mask_still_not_parsed, 'Date'], errors='coerce'
            )

        min_date = df_trackman['Date_parsed'].min()
        max_date = df_trackman['Date_parsed'].max()
        print(f"Trackman file Date range:")
        print(f"  Min Date: {min_date}")
        print(f"  Max Date: {max_date}")
        # Replace Date column with parsed if all parsed values present
        if df_trackman['Date_parsed'].notna().all():
            df_trackman['Date'] = df_trackman['Date_parsed']
        df_trackman.drop(columns=['Date_parsed'], inplace=True)
    else:
        print("No 'Date' column found in Trackman CSV.")

except Exception as e:
    print(f"Error loading or processing Trackman CSV: {e}")



In [ ]:
# Append the specified CSV file to the master file and export

master_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv"
csv_to_append_path = r"C:\Users\TrevorWhite\Downloads\20251102-UofSanDiego-Private-1_unverified.csv"

import os
import pandas as pd

# Load the master file if it exists, else create an empty DataFrame
if os.path.exists(master_path):
    master_df = pd.read_csv(master_path)
else:
    master_df = pd.DataFrame()

# Load the CSV to append
if os.path.exists(csv_to_append_path):
    append_df = pd.read_csv(csv_to_append_path)
else:
    raise FileNotFoundError(f"File not found: {csv_to_append_path}")

# Append (i.e., add rows of append_df below master_df)
updated_df = pd.concat([master_df, append_df], ignore_index=True)

# Export
updated_df.to_csv(master_path, index=False)
print(f"Appended '{csv_to_append_path}' to master file and exported to {master_path}")

In [ ]:
import os
import pandas as pd

# Define paths
directory = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs"
processed_folder = os.path.join(directory, 'processed')
master_file_path = os.path.join(directory, 'usd_baseball_TM_master_file.csv')

# Initialize an empty list to store new dataframes
csv_dataframes = []

# Loop through all files in the processed folder
for filename in os.listdir(processed_folder):
    file_path = os.path.join(processed_folder, filename)
    if filename.endswith(".csv"):
        try:
            # Read the CSV file and append it to the list
            df = pd.read_csv(file_path)
            csv_dataframes.append(df)
            print(f"Successfully read {filename}")
        except Exception as e:
            print(f"Error reading {filename}: {e}")

# Combine all new dataframes into a single dataframe if there are any new files
if csv_dataframes:
    new_combined_df = pd.concat(csv_dataframes, ignore_index=True)

    # Save the combined dataframe directly to the master file
    # This overwrites any existing master file, effectively truncating it.
    try:
        new_combined_df.to_csv(master_file_path, index=False)
        print(f"Master CSV file overwritten and saved at {master_file_path}")
    except Exception as e:
        print(f"Error saving the updated CSV: {e}")
else:
    print("No new CSV files found in the processed folder.")


In [ ]:
import os
import pandas as pd
import re
import shutil

# Define the directory and processed folder paths
directory = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs"
processed_folder = os.path.join(directory, 'processed')
master_file_url = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/usd_baseball_TM_master_file.csv"
master_file_path = os.path.join(directory, 'usd_baseball_TM_master_file.csv')

# Ensure the processed folder exists
os.makedirs(processed_folder, exist_ok=True)

# Regex pattern for filenames
file_pattern = r"\d{8}-UofSanDiego-(Private|BP)-\d+_unverified\.csv"

# Download the master file from the URL
try:
    master_df = pd.read_csv(master_file_url)
    print("Master file successfully downloaded from the provided URL.")
except Exception as e:
    print(f"Error downloading the master file: {e}")
    master_df = pd.DataFrame()  # Create an empty DataFrame as a fallback

# Initialize an empty list to store new dataframes
csv_dataframes = []

# Loop through files in the directory
for filename in os.listdir(directory):
    if re.match(file_pattern, filename):  # Check if the filename matches the pattern
        file_path = os.path.join(directory, filename)
        
        try:
            # Read and append CSV data
            df = pd.read_csv(file_path)
            csv_dataframes.append(df)
            print(f"Successfully read {filename}")
            
            # Move processed file to the processed folder
            shutil.move(file_path, os.path.join(processed_folder, filename))
            print(f"Moved {filename} to {processed_folder}")
        except Exception as e:
            print(f"Error processing {filename}: {e}")

# Combine matched data with the master file
if csv_dataframes:
    new_data = pd.concat(csv_dataframes, ignore_index=True)
    
    try:
        # Combine new data with the master file
        if not master_df.empty:
            combined_df = pd.concat([master_df, new_data], ignore_index=True)
            print("Master file combined with new data.")
        else:
            combined_df = new_data
            print("No valid master file found. Using only new data.")
        
        # Save the updated master file locally
        combined_df.to_csv(master_file_path, index=False)
        print(f"Master file updated and saved at: {master_file_path}")
    except Exception as e:
        print(f"Error updating the master file: {e}")
else:
    print("No new files matched the criteria.")


In [ ]:
import joblib

# Load the model from the file
best_model = joblib.load('best_xgboost_model.pkl')
print("Model loaded successfully!")

# # Use the model to make predictions
# y_pred = best_model.predict(X_test)
# print(y_pred)


In [ ]:
library(ggplot2)

# Sample data
set.seed(123)
df <- data.frame(
  x = runif(1000, 0, 10),
  y = runif(1000, 0, 10),
  z = runif(1000, -5, 5) # Third variable
)

# Plot
ggplot(df, aes(x, y, fill = z)) +
  geom_raster() + # Raster for dense grid
  scale_fill_gradient2(low = "blue", mid = "white", high = "red", midpoint = 0) +
  labs(title = "Heatmap colored by z-variable",
       fill = "Z Value") +
  theme_minimal()


In [ ]:
import pandas as pd

# Correct file path to your Excel file
file_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\playerheights.xlsx"

# Load the Excel file
try:
    df = pd.read_excel(file_path)
except FileNotFoundError as e:
    print(f"Error: {e}")
    raise

# Function to convert height to decimal format
def height_to_decimal(height):
    if pd.isna(height):
        return None
    try:
        feet, inches = height.split("'")
        feet = int(feet.strip())
        inches = int(inches.replace('"', '').strip())
        return round(feet + inches / 12.0, 2)
    except ValueError:
        return None

# Function to reformat player name to "Last, First" format
def format_name(name):
    if pd.isna(name):
        return None
    try:
        parts = name.split()
        if len(parts) < 2:
            return name  # Return as-is if name is incomplete
        first_name = parts[0]
        last_name = parts[-1]
        return f"{last_name}, {first_name}"
    except Exception as e:
        return name  # Return original name on failure

# Clean the height column
if 'Height' in df.columns:
    df['Height'] = df['Height'].apply(height_to_decimal)
else:
    print("Error: The 'Height' column was not found in the Excel file.")

# Reformat the name column
if 'Player Name' in df.columns:
    df['Player Name'] = df['Player Name'].apply(format_name)
else:
    print("Error: The 'Player Name' column was not found in the Excel file.")

# Select cleaned columns
df_cleaned = df[['Player Name', 'Height']] if 'Height' in df.columns and 'Player Name' in df.columns else None

# # Save the cleaned DataFrame to a new file for verification
# output_file = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\cleaned_playerheights.xlsx"
# if df_cleaned is not None:
#     df_cleaned.to_excel(output_file, index=False)
#     print(f"Cleaned data saved to {output_file}")


In [ ]:
Tilt convert to minutes like this but instead of pitch hand use avg relside - = L + = R
import pandas as pd
import numpy as np
from datetime import datetime

##original file
"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv"



def signed_minutes_from_12(row):
    time_str = row['tilt']
    pitch_hand = row['relside']   -------instead of this change it to relside, if relside is positive it is R
    # Check for missing values
    if pd.isnull(time_str) or pd.isnull(pitch_hand):
        return np.nan
    # Convert to string and strip whitespace
    time_str = str(time_str).strip()
    pitch_hand = str(pitch_hand).strip()
    if time_str == '':
        return np.nan
    try:
        # Parse the time string into a datetime object
        time_obj = datetime.strptime(time_str, '%I:%M')
        # Convert time to minutes since 12:00
        total_minutes = (time_obj.hour % 12) * 60 + time_obj.minute  # 0 to 719
        # Calculate signed difference based on pitcher's hand
        if total_minutes == 0:
            difference = 0
        elif total_minutes < 360:
            # Times after 12:00 up to 5:59
            if pitch_hand == 'R':      -------instead of this change it to relside, if relside is positive it is R
                difference = total_minutes  # Positive minutes
            else:
                difference = -total_minutes  # Negative minutes
        else:
            # Times from 6:00 up to 11:59
            minutes_to_12 = 720 - total_minutes
            if pitch_hand == 'R':    -------instead of this change it to relside, if relside is positive it is R
                difference = -minutes_to_12  # Negative minutes
            else:
                difference = minutes_to_12  # Positive minutes
        return difference
    except ValueError:
        # If parsing fails, return NaN
        return np.nan

# Apply the function to the DataFrame row-wise
combined_df['minutes_past_12'] = combined_df.apply(signed_minutes_from_12, axis=1)

# Display the updated DataFrame
print(combined_df[['player_id', 'hawkeye_measured_clock_label', 'pitch_hand_y', 'minutes_past_12']])




avg all of these
minutespast12 RelHeight	RelSide	Extension	VertBreak	InducedVertBreak	HorzBreak	PlateLocHeight	PlateLocSide spinrate relspeed




load

"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\playerheights.xlsx"


clean player name like in the previous code so it is last, first


join "C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\playerheights.xlsx" on height.[Player Name] = tm.Pitcher



height


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# File paths
master_file_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv"
player_heights_file_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\playerheights.xlsx"

# Load the master file
master_df = pd.read_csv(master_file_path)

# Load the player heights file
player_heights_df = pd.read_excel(player_heights_file_path)

# Clean player names in the heights file
def format_name(name):
    if pd.isnull(name):
        return None
    try:
        parts = name.split()
        if len(parts) < 2:
            return name  # Return as-is if name is incomplete
        first_name = parts[0]
        last_name = parts[-1]
        return f"{last_name}, {first_name}"
    except Exception:
        return name

player_heights_df['Player Name'] = player_heights_df['Player Name'].apply(format_name)

# Determine dominant pitch type for each pitcher
# Function to determine the dominant pitch type among 'Four-Seam' and 'Sinker'
def get_dominant_pitch_type(pitches):
    # Filter pitches to only 'Four-Seam' and 'Sinker'
    counts = pitches[pitches.isin(['Four-Seam', 'Sinker'])].value_counts()
    if counts.empty:
        return None  # No 'Four-Seam' or 'Sinker' pitches found
    else:
        # Return the pitch type with the highest count
        return counts.idxmax()

# Apply the function to each pitcher
dominant_pitch_types = master_df.groupby('Pitcher')['AutoPitchType'].apply(get_dominant_pitch_type).reset_index(name='DominantPitchType')

# Merge the dominant pitch types back to the main DataFrame
master_df = master_df.merge(dominant_pitch_types, on='Pitcher', how='left')

# Filter the DataFrame where AutoPitchType equals DominantPitchType
master_df = master_df[master_df['AutoPitchType'] == master_df['DominantPitchType']]

# Display the head of the resulting DataFrame
master_df.head()

# Convert tilt to signed minutes based on relside
def signed_minutes_from_12(row):
    time_str = row['Tilt']
    relside = row['RelSide']
    
    # Check for missing values
    if pd.isnull(time_str) or pd.isnull(relside):
        return np.nan
    
    try:
        # Parse the time string into a datetime object
        time_obj = datetime.strptime(time_str.strip(), '%I:%M')
        
        # Calculate total minutes based on RelSide
        if relside > 0:
            # Positive RelSide: minutes past 12
            total_minutes = ((time_obj.hour % 12) * 60) + time_obj.minute
            return total_minutes
        else:
            # Negative RelSide: minutes before 12
            total_minutes = ((12 - (time_obj.hour % 12)) % 12) * 60 - time_obj.minute
            # Adjust for negative total_minutes
            if total_minutes == 0:
                total_minutes = 720  # Edge case for 12:00
            return total_minutes
    except ValueError:
        # If parsing fails, return NaN
        return np.nan


# Apply the function to calculate minutes past 12
master_df['minutes_past_12'] = master_df.apply(signed_minutes_from_12, axis=1)

# Columns to average
columns_to_average = [
    'minutes_past_12', 'RelHeight', 'RelSide', 'Extension', 
    'InducedVertBreak', 'HorzBreak', 'PlateLocHeight', 'PlateLocSide',
    'SpinRate', 'RelSpeed'
]

# # Select only the relevant columns for averaging
columns_to_keep = ['Pitcher', 'DominantPitchType'] + columns_to_average
grouped_averages = master_df[columns_to_keep].groupby(['Pitcher', 'DominantPitchType']).mean().reset_index()

# Merge the aggregated averages with player heights
final_df = grouped_averages.merge(player_heights_df, left_on='Pitcher', right_on='Player Name', how='left')

# # Display the first 100 rows
final_df.head(100)

##platelocside is positive to rhb


In [ ]:


# Step 1: Apply the conditional logic based on the original 'relside'
final_df.loc[final_df['RelSide'] < 0, ['HorzBreak', 'PlateLocSide']] *= -1

# Step 2: Convert 'relside' to absolute values
final_df['RelSide'] = final_df['RelSide'].abs()






In [ ]:
# Ensure we handle cases where the height format is invalid or missing
final_df['Height_decimal'] = final_df['Height'].str.extract(r"(\d+)'(\d+)") \
    .apply(lambda x: (int(x[0]) + int(x[1]) / 12) if pd.notnull(x[0]) and pd.notnull(x[1]) else None, axis=1)

final_df.head()

In [ ]:
# Create 'FF' column (True if DominantPitchType is 'Four-Seam', False otherwise)
final_df['FF'] = final_df['DominantPitchType'] == 'Four-Seam'

# Create 'SI' column (True if DominantPitchType is 'Sinker', False otherwise)
final_df['SI'] = final_df['DominantPitchType'] == 'Sinker'

In [ ]:
# Create derived interaction features
final_df['height_Ext'] = final_df['Height_decimal'] * final_df['Extension']
final_df['height_zRel'] = final_df['RelHeight'] * final_df['Height_decimal']
final_df['height_xRel'] = final_df['RelSide'] * final_df['Height_decimal']
final_df['tilt_min_xRel'] = final_df['minutes_past_12'] * final_df['RelSide']
final_df['tilt_min_zRel'] = final_df['minutes_past_12'] * final_df['RelHeight']
final_df['ivb_zRel'] = final_df['InducedVertBreak'] * final_df['RelHeight']
final_df['ArmSideMovement_xRel'] = final_df['RelSide'] * final_df['HorzBreak']
final_df['spin_ivb'] = final_df['SpinRate'] * final_df['InducedVertBreak']
final_df['spin_ArmSideMovement'] = final_df['SpinRate'] * final_df['HorzBreak']
final_df['Ext_zRel'] = final_df['Extension'] * final_df['RelHeight']
final_df['Ext_xRel'] = final_df['Extension'] * final_df['RelSide']
final_df['ff_hb'] = final_df['HorzBreak'] * final_df['FF']
final_df['ff_ivb'] = final_df['InducedVertBreak'] * final_df['FF']
final_df['ivb_plate_z'] = final_df['InducedVertBreak'] * final_df['PlateLocHeight']
final_df['ArmSideMovement_plate_x'] = final_df['HorzBreak'] * final_df['PlateLocSide']
final_df['FF_plate_z'] = final_df['PlateLocHeight'] * final_df['FF']
final_df['FF_plate_x'] = final_df['PlateLocSide'] * final_df['FF']
final_df['spin_plate_z'] = final_df['SpinRate'] * final_df['PlateLocHeight']
final_df['spin_plate_x'] = final_df['SpinRate'] * final_df['PlateLocSide']

# Rename the columns to match the model's required feature names
model_ready_df = final_df.rename(columns={
    'RelSide': 'xRel',
    'RelHeight': 'release_pos_z',
    'Extension': 'release_extension',
    'RelSpeed': 'velocity',
    'InducedVertBreak': 'ivb',
    'HorzBreak': 'hb_cleaned',
    'PlateLocHeight': 'plate_z',
    'PlateLocSide': 'plate_x',
    'SpinRate': 'spin_rate',
    'Height_decimal': 'height',
})


In [ ]:
import xgboost as xgb

# Load the model
model_path = r"C:\\Users\\TrevorWhite\\OneDrive - Good360\\Documents\\Python Scripts\\best_xgboost_model.json"
xgb_model = xgb.Booster()
xgb_model.load_model(model_path)

# Select only the required columns for the model
required_features = [
    "xRel", "release_pos_z", "FF", "release_extension", "height", "velocity", "minutes_past_12",
    "ivb", "hb_cleaned", "spin_rate", "plate_x", "plate_z", "height_Ext", "height_zRel", "height_xRel",
    "tilt_min_xRel", "tilt_min_zRel", "ivb_zRel", "ArmSideMovement_xRel", "spin_ivb", "spin_ArmSideMovement",
    "Ext_zRel", "Ext_xRel", "ff_hb", "ff_ivb", "ivb_plate_z", "ArmSideMovement_plate_x", "FF_plate_z",
    "FF_plate_x", "spin_plate_z", "spin_plate_x"
]

# Keep only the required columns
analysis_df = model_ready_df[required_features].dropna()

# Convert the DataFrame to DMatrix for prediction
dmatrix = xgb.DMatrix(analysis_df)

# Make predictions
analysis_df['armangle_prediction'] = xgb_model.predict(dmatrix)

In [ ]:
analysis_df = analysis_df.join(final_df['Pitcher'], how='inner')
analysis_df.head()



In [ ]:
analysis_df.head(29)

In [ ]:
# Save the resulting DataFrame to a CSV file in the Downloads folder
output_path = r"C:\\Users\\TrevorWhite\\Downloads\\armangle_predictions.csv"
analysis_df[['Pitcher', 'armangle_prediction']].to_csv(output_path, index=False)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.scatter(analysis_df['release_pos_z'], analysis_df['armangle_prediction'], alpha=0.6)
for i, row in analysis_df.iterrows():
    plt.text(row['release_pos_z'], row['armangle_prediction'], str(row['Pitcher']), fontsize=8, alpha=0.7)
plt.title('Scatter Plot of Release Position Z vs Arm Angle Prediction')
plt.xlabel('Release Position Z')
plt.ylabel('Arm Angle Prediction')
plt.grid(True)
plt.show()

In [ ]:
# Apply the dominant pitch type to all rows for each pitcher
master_df['DominantPitchType'] = master_df.groupby('Pitcher')['AutoPitchType'].transform(calculate_dominant_pitch_type)

#  # Filter master DataFrame based on dominant pitch type
# master_df = master_df[master_df['AutoPitchType'] == master_df['DominantPitchType']]



# # Convert tilt to signed minutes based on relside
# def signed_minutes_from_12(row):
#     time_str = row['Tilt']
#     relside = row['RelSide']
    
#     # Check for missing values
#     if pd.isnull(time_str) or pd.isnull(relside):
#         return np.nan
    
#     try:
#         # Parse the time string into a datetime object
#         time_obj = datetime.strptime(time_str.strip(), '%I:%M')
#         # Convert time to minutes since 12:00
#         total_minutes = (time_obj.hour % 12) * 60 + time_obj.minute
        
#         # Determine signed minutes based on relside
#         if relside > 0:  # Positive relside: count minutes past 12
#             return total_minutes
#         else:  # Negative relside: count minutes before 12
#             return -total_minutes
#     except ValueError:
#         # If parsing fails, return NaN
#         return np.nan

# # Apply the function to calculate minutes past 12
# master_df['minutes_past_12'] = master_df.apply(signed_minutes_from_12, axis=1)

# # Columns to average
# columns_to_average = [
#     'minutes_past_12', 'RelHeight', 'RelSide', 'Extension', 
#     'InducedVertBreak', 'HorzBreak', 'PlateLocHeight', 'PlateLocSide',
#     'SpinRate', 'RelSpeed'
# ]

# # Select only the relevant columns for averaging
# columns_to_keep = ['Pitcher', 'DominantPitchType'] + columns_to_average
# grouped_averages = master_df[columns_to_keep].groupby(['Pitcher', 'DominantPitchType']).mean().reset_index()

# # Merge the aggregated averages with player heights
# final_df = grouped_averages.merge(player_heights_df, left_on='Pitcher', right_on='Player Name', how='left')

# # Display the first 100 rows
# print(final_df.head(100))


In [ ]:
pd.set_option('display.max_columns', None)

pd.set_option('display.max_rows', None)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
import seaborn as sns

# Function to calculate EventRV based on PitchCall and PlayResult
def calculate_event_rv(row):
    event_rv_dict = {
        'HomeRun': 1.374328827219,
        'Triple': 1.05755624961515,
        'Double': 0.766083122898271,
        'Single': 0.467292970729251,
        'FieldersChoice': -0.1955687665555,
        'Out': -0.1955687665555,
        'Sac': -0.195,
        'Error': -0.195,
        'Ball': 0.0636883289483747,
        'HitByPitch': 0.0636883289483747,
        'Foul': -0.0380502742575014,
        'StrikeCalled': -0.065092516089806,
        'StrikeSwinging': -0.118124935770601,
    }
    if row['PitchCall'] == 'InPlay':
        return event_rv_dict.get(row['PlayResult'], 0)
    else:
        return event_rv_dict.get(row['PitchCall'], 0)

# Apply EventRV calculation to dataframe
df['EventRV'] = df.apply(calculate_event_rv, axis=1)

# Function to get pitch scatter data based on pitch type and batter side
def get_pitch_scatter_data(df, Pitcher, pitch_type, batter_side):
    if pitch_type == "Breaking Ball":
        pitch_df = df.loc[(df['Pitcher'] == Pitcher) & 
                          (df['AutoPitchType'].isin(["Slider", "Curveball"])) & 
                          (df['BatterSide'] == batter_side)]
    elif pitch_type == "Fastball":
        pitch_df = df.loc[(df['Pitcher'] == Pitcher) & 
                          (df['AutoPitchType'].isin(["Fastball", "Four-Seam", "Sinker", "Cutter"])) & 
                          (df['BatterSide'] == batter_side)]
    elif pitch_type == "Offspeed":
        pitch_df = df.loc[(df['Pitcher'] == Pitcher) & 
                          (df['AutoPitchType'].isin(["ChangeUp", "Splitter"])) & 
                          (df['BatterSide'] == batter_side)]
    
    pitch_df = pitch_df.dropna(subset=['PlateLocSide', 'PlateLocHeight', 'EventRV'])
    if pitch_df.empty:
        print(f"Not enough valid data for {Pitcher}'s {pitch_type} against {batter_side} batters.")
        return None
    return pitch_df[['PlateLocSide', 'PlateLocHeight', 'EventRV']]

# Load CSV data
file_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)

# Check column names to ensure correct column references
print("Column names in the dataframe: ", df.columns)
# # Standardize column capitalization
# df.columns = [col.strip().capitalize() for col in df.columns]

# Define the plate vertices
plate_vertices = [(-0.83, 0.1), (0.83, 0.1), (0.65, 0.25), (0, 0.5), (-0.65, 0.25)]
plate = Polygon(plate_vertices, closed=True, linewidth=1, edgecolor='k', facecolor='none')

# Function to create scatter plots for all pitchers
def plot_scatter_for_all_pitchers(df):
    for pitcher_name in df['Pitcher'].unique():
        fig, gs = plt.subplots(3, 3, figsize=(15, 10))  # Adjust the figure and grid size
        
        # Retrieve pitch location data for each combination
        breakingball_LHH_data = get_pitch_scatter_data(df, pitcher_name, 'Breaking Ball', 'Left')
        breakingball_RHH_data = get_pitch_scatter_data(df, pitcher_name, 'Breaking Ball', 'Right')
        offspeed_LHH_data = get_pitch_scatter_data(df, pitcher_name, 'Offspeed', 'Left')
        offspeed_RHH_data = get_pitch_scatter_data(df, pitcher_name, 'Offspeed', 'Right')
        fastball_LHH_data = get_pitch_scatter_data(df, pitcher_name, 'Fastball', 'Left')
        fastball_RHH_data = get_pitch_scatter_data(df, pitcher_name, 'Fastball', 'Right')

        norm = plt.Normalize(-.2, 1.4)
        cmap = plt.cm.RdYlGn_r  # Reverse the colormap for better contrast

        # Fastball RHH
        if fastball_RHH_data is not None:
            ax6 = fig.add_subplot(gs[1, 0])
            ax6.set_title(f'Fastball Locations (RHH) - {pitcher_name}', fontsize=16)
            scatter = ax6.scatter(fastball_RHH_data['PlateLocSide'], fastball_RHH_data['PlateLocHeight'], 
                                  c=fastball_RHH_data['EventRV'], cmap=cmap, norm=norm, s=80, alpha=0.5)
            ax6.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
            ax6.add_patch(plate)
            ax6.set_xlim(-2, 2)
            ax6.set_ylim(0, 4.5)

        # Fastball LHH
        if fastball_LHH_data is not None:
            ax7 = fig.add_subplot(gs[2, 0])
            scatter = ax7.scatter(fastball_LHH_data['PlateLocSide'], fastball_LHH_data['PlateLocHeight'], 
                                  c=fastball_LHH_data['EventRV'], cmap=cmap, norm=norm, s=80, alpha=0.5)
            ax7.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
            ax7.add_patch(plate)
            ax7.set_xlim(-2, 2)
            ax7.set_ylim(0, 4.5)

        # Breaking Ball RHH
        if breakingball_RHH_data is not None:
            ax8 = fig.add_subplot(gs[1, 1])
            ax8.set_title(f'Breaking Ball Locations (RHH) - {pitcher_name}', fontsize=16)
            scatter = ax8.scatter(breakingball_RHH_data['PlateLocSide'], breakingball_RHH_data['PlateLocHeight'], 
                                  c=breakingball_RHH_data['EventRV'], cmap=cmap, norm=norm, s=80, alpha=0.5)
            ax8.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
            ax8.add_patch(plate)
            ax8.set_xlim(-2, 2)
            ax8.set_ylim(0, 4.5)

        # Breaking Ball LHH
        if breakingball_LHH_data is not None:
            ax9 = fig.add_subplot(gs[2, 1])
            scatter = ax9.scatter(breakingball_LHH_data['PlateLocSide'], breakingball_LHH_data['PlateLocHeight'], 
                                  c=breakingball_LHH_data['EventRV'], cmap=cmap, norm=norm, s=80, alpha=0.5)
            ax9.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
            ax9.add_patch(plate)
            ax9.set_xlim(-2, 2)
            ax9.set_ylim(0, 4.5)

        # Offspeed RHH
        if offspeed_RHH_data is not None:
            ax10 = fig.add_subplot(gs[1, 2])
            ax10.set_title(f'Offspeed Locations (RHH) - {pitcher_name}', fontsize=16)
            scatter = ax10.scatter(offspeed_RHH_data['PlateLocSide'], offspeed_RHH_data['PlateLocHeight'], 
                                   c=offspeed_RHH_data['EventRV'], cmap=cmap, norm=norm, s=80, alpha=0.5)
            ax10.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
            ax10.add_patch(plate)
            ax10.set_xlim(-2, 2)
            ax10.set_ylim(0, 4.5)

        # Offspeed LHH
        if offspeed_LHH_data is not None:
            ax11 = fig.add_subplot(gs[2, 2])
            scatter = ax11.scatter(offspeed_LHH_data['PlateLocSide'], offspeed_LHH_data['PlateLocHeight'], 
                                   c=offspeed_LHH_data['EventRV'], cmap=cmap, norm=norm, s=80, alpha=0.5)
            ax11.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, linewidth=1, edgecolor='k', facecolor='none'))
            ax11.add_patch(plate)
            ax11.set_xlim(-2, 2)
            ax11.set_ylim(0, 4.5)

        # Show the plot for the pitcher
        plt.tight_layout()
        plt.show()

# Call the function to plot for all pitchers
plot_scatter_for_all_pitchers(df)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime
import os

# Load the CSV file
file_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)

# Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

# Convert 'Tilt' column from HH:MM format to float (1:45 -> 1.75)
def convert_tilt_to_float(tilt_value):
    try:
        if isinstance(tilt_value, str) and ':' in tilt_value:
            hours, minutes = tilt_value.split(':')
            return float(hours) + float(minutes) / 60
        return np.nan
    except Exception as e:
        print(f"Error converting Tilt: {e}")
        return np.nan

# Apply Tilt conversion
df['Tilt_float'] = df['Tilt'].apply(convert_tilt_to_float)

# Identify in-zone pitches based on PlateLocSide and PlateLocHeight
df['Inzone'] = df.apply(
    lambda row: -0.83 <= row['Platelocside'] <= 0.83 and 1.5 <= row['Platelocheight'] <= 3.6, axis=1)

# Define pitch categories based on initial pitch types
pitch_categories = {
    "Breaking Ball": ["Slider", "Curveball"],
    "Fastball": ["Fastball", "Four-Seam", "Sinker", "Cutter"],
    "Offspeed": ["ChangeUp", "Splitter"]
}

# Function to categorize pitch types into broader groups
def categorize_pitch_type(pitch_type):
    for category, pitches in pitch_categories.items():
        if pitch_type in pitches:
            return category
    return None

# Create a new column 'Pitchcategory' to categorize pitches
df['Pitchcategory'] = df['Autopitchtype'].apply(categorize_pitch_type)

# Set up the color palette based on pitch type
pitch_types = df['Autopitchtype'].unique()
palette = sns.color_palette('hsv', len(pitch_types))
color_map = dict(zip(pitch_types, palette))

# Function to safely create directories if they don't exist
def safe_create_dir(path):
    os.makedirs(path, exist_ok=True)

# Function to create scatter plots for each pitcher with pitch locations, separated by batter side
def plot_pitch_locations_by_pitcher(pitcher_data, pitcher_name, pdf):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    batter_sides = ['Right', 'Left']

    # Define the vertices of the plate. The vertices are based on a typical home plate outline.
    plate_vertices = [(-0.83, 0.1), (0.83, 0.1), (0.65, 0.25), (0, 0.5), (-0.65, 0.25)]

    for i, batter_side in enumerate(batter_sides):
        side_data = pitcher_data[pitcher_data['Batterside'] == batter_side]

        sns.scatterplot(
            data=side_data,
            x='Platelocside',
            y='Platelocheight',
            hue='Autopitchtype',
            palette=color_map,
            s=100,
            edgecolor='black',
            ax=axes[i]
        )

        # Add the strike zone as a rectangle
        axes[i].add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, edgecolor='black', facecolor='none'))

        # Create a new home plate polygon for each subplot to avoid reuse errors
        plate = Polygon(plate_vertices, closed=True, linewidth=1, edgecolor='k', facecolor='none')
        axes[i].add_patch(plate)

        axes[i].set_title(f'{pitcher_name} vs {batter_side} Handed Batters', fontsize=14)
        axes[i].set_xlim(-2.5, 2.5)
        axes[i].set_ylim(0, 5)
        axes[i].set_xlabel('Horizontal Plate Location')
        axes[i].set_ylabel('Vertical Plate Location')
        axes[i].legend(title='Pitch Type', bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    pdf.savefig(fig)
    plt.close()


# Function to create polar plots for each pitcher
def create_polar_plots(pitcher_data, pitcher_name, pdf):
    pitcher_data = pitcher_data.dropna(subset=['Tilt_float'])
    if pitcher_data.empty:
        print(f"No data with Tilt information for {pitcher_name}. Skipping polar plots.")
        return
    tilt_radians = np.radians(pitcher_data['Tilt_float'] * 30)

    fig, axs = plt.subplots(1, 2, subplot_kw={'projection': 'polar'}, figsize=(12, 6))
    axs[0].set_theta_zero_location('N')
    axs[0].set_theta_direction(-1)
    axs[1].set_theta_zero_location('N')
    axs[1].set_theta_direction(-1)

    sc1 = axs[0].scatter(tilt_radians, pitcher_data['Relspeed'], c=pitcher_data['Autopitchtype'].map(color_map), s=100)
    axs[0].set_title('Velocity vs. Tilt', fontsize=14)
    axs[0].set_ylim(pitcher_data['Relspeed'].min() - 5, pitcher_data['Relspeed'].max() + 5)
    axs[0].set_xticks(np.radians(np.arange(0, 360, 30)))
    axs[0].set_xticklabels(['12', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11'])

    sc2 = axs[1].scatter(tilt_radians, pitcher_data['Spinrate'], c=pitcher_data['Autopitchtype'].map(color_map), s=100)
    axs[1].set_title('Spin Rate vs. Tilt', fontsize=14)
    axs[1].set_ylim(pitcher_data['Spinrate'].min() - 500, pitcher_data['Spinrate'].max() + 500)
    axs[1].set_xticks(np.radians(np.arange(0, 360, 30)))
    axs[1].set_xticklabels(['12', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11'])

    plt.suptitle(f'Pitcher: {pitcher_name}', fontsize=16)
    handles = [plt.Line2D([0], [0], marker='o', color=color_map[pt], markersize=10, linestyle='') for pt in pitch_types]
    plt.legend(handles, pitch_types, title='Pitch Type', bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    pdf.savefig(fig)
    plt.close(fig)

# Function to create release point plot for each pitcher
def create_release_plot(pitcher_data, pitcher_name, pdf):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        data=pitcher_data,
        x='Relside',
        y='Relheight',
        hue='Autopitchtype',
        palette=color_map,
        s=100,
        edgecolor='black'
    )
    plt.xlim(pitcher_data['Relside'].min() - 1, pitcher_data['Relside'].max() + 1)
    plt.ylim(pitcher_data['Relheight'].min() - 1, pitcher_data['Relheight'].max() + 1)
    plt.xlabel('Horizontal Release Point')
    plt.ylabel('Vertical Release Point')
    plt.title(f'Release Point for {pitcher_name}', fontsize=16)
    plt.legend(title='Pitch Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    pdf.savefig()
    plt.close()
    
# Function to calculate pitch metrics for each pitch type
def calculate_pitch_metrics(pitcher_data):
    # Count pitches for each type
    pitch_type_counts = pitcher_data['Autopitchtype'].value_counts().rename('Count')

    # Calculate averages for each pitch type (excluding Tilt)
    avg_cols = ['Relspeed', 'Spinrate', 'Inducedvertbreak', 'Horzbreak',
                'Relheight', 'Relside', 'Extension', 'Vertapprangle', 'Horzapprangle']
    pitch_type_averages = pitcher_data.groupby('Autopitchtype')[avg_cols].mean().round(1)

    # Calculate strike percentages
    strikes = pitcher_data[pitcher_data['Pitchcall'].isin(['StrikeCalled', 'StrikeSwinging', 'FoulBall', 'InPlay'])]
    strike_percentages = (strikes.groupby('Autopitchtype').size() / pitch_type_counts * 100).rename('Strike %').round(1)

    # Calculate whiff percentages
    swinging_strikes = pitcher_data[pitcher_data['Pitchcall'] == 'StrikeSwinging'].groupby('Autopitchtype').size()
    total_swings = pitcher_data[pitcher_data['Pitchcall'].isin(['StrikeSwinging', 'FoulBall', 'InPlay'])].groupby('Autopitchtype').size()
    whiff_percentages = (swinging_strikes / total_swings * 100).rename('Whiff %').fillna(0).round(1)

    # Calculate max velocity
    max_velocity = pitcher_data.groupby('Autopitchtype')['Relspeed'].max().rename('Max velo').round(1)

    # Calculate InZone %
    in_zone_total = pitcher_data[pitcher_data['Inzone']].groupby('Autopitchtype').size()
    in_zone_percentage = (in_zone_total / pitch_type_counts * 100).rename('InZone %').fillna(0).round(1)

    # Calculate InZone Whiff %
    in_zone_swinging_strikes = pitcher_data[(pitcher_data['Inzone']) & (pitcher_data['Pitchcall'] == 'StrikeSwinging')].groupby('Autopitchtype').size()
    in_zone_swings = pitcher_data[pitcher_data['Inzone'] & pitcher_data['Pitchcall'].isin(['StrikeSwinging', 'FoulBall', 'InPlay'])].groupby('Autopitchtype').size()
    in_zone_whiff_percentages = (in_zone_swinging_strikes / in_zone_swings * 100).rename('InZone Whiff %').fillna(0).round(1)

    # Calculate Chase %
    out_zone_swings = pitcher_data[(~pitcher_data['Inzone']) & pitcher_data['Pitchcall'].isin(['StrikeSwinging', 'FoulBall', 'InPlay'])].groupby('Autopitchtype').size()
    total_out_zone_pitches = pitcher_data[~pitcher_data['Inzone']].groupby('Autopitchtype').size()
    chase_percentage = (out_zone_swings / total_out_zone_pitches * 100).rename('Chase %').fillna(0).round(1)

    # Join calculated stats
    metrics_df = (pitch_type_counts.to_frame()
                  .join(max_velocity)
                  .join(pitch_type_averages)
                  .join(strike_percentages)
                  .join(whiff_percentages)
                  .join(in_zone_percentage)
                  .join(in_zone_whiff_percentages)
                  .join(chase_percentage)
                  .fillna(0))

    correct_order = ['P',  'Max velo', 'AVG velo', 'Spinrate', 'IVB', 'HB',
                     'yRel', 'xRel', 'Ext.', 'VAA', 'HAA',
                     'Strike %', 'Whiff %', 'InZone %', 'InZone Whiff %', 'Chase %']
    metrics_df.columns = correct_order
    return metrics_df

# Function to create a table plot for each pitcher
def create_table_plot(pitcher_data, pitcher_name, pdf):
    # Calculate the metrics using the existing function
    metrics_df = calculate_pitch_metrics(pitcher_data)
    
    # Manually set the new column names based on the specified correct order
    correct_order = ['P',  'Max velo', 'AVG velo', 'Spinrate', 'IVB', 'HB',
                     'yRel', 'xRel', 'Ext.', 'VAA', 'HAA',
                     'Strike %', 'Whiff %', 'InZone %', 'InZone Whiff %', 'Chase %']
    
    # Ensure the DataFrame has the same number of columns as correct_order
    if len(metrics_df.columns) == len(correct_order):
        metrics_df.columns = correct_order
    else:
        print("Warning: Column count mismatch. Please verify the 'calculate_pitch_metrics' output.")

    # Create the table figure
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.axis('off')

    # Generate the table
    table = ax.table(cellText=metrics_df.values,
                     colLabels=metrics_df.columns,
                     rowLabels=metrics_df.index,
                     cellLoc='center',
                     loc='center')

    # Customize the table appearance
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.2)

    # Set the title and save the figure to the PDF
    plt.title(f'Pitch Metrics for {pitcher_name}', fontsize=16)
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

# Function to create a break plot for each pitcher
def create_break_plot(pitcher_data, pitcher_name, pdf):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=pitcher_data,
        x='Horzbreak',
        y='Inducedvertbreak',
        hue='Autopitchtype',
        palette=color_map,
        s=100,
        edgecolor='black'
    )
    
    # Set the axis limits to always show from -25 to 25
    plt.xlim(-25, 25)
    plt.ylim(-25, 25)
    
    # Draw reference lines at zero
    plt.axvline(0, color='grey', linestyle='--')
    plt.axhline(0, color='grey', linestyle='--')
    
    # Determine arm side and glove side positions based on the pitcher's throwing hand
    pitcher_throws = pitcher_data['Pitcherthrows'].iloc[0]  # Assuming all rows have the same pitcher
    if pitcher_throws == 'Right':
        arm_side_x = 20
        glove_side_x = -20
    elif pitcher_throws == 'Left':
        arm_side_x = -20
        glove_side_x = 20
    else:
        arm_side_x = glove_side_x = None  # Handle unexpected values gracefully

    # Add "Arm Side" and "Glove Side" labels
    if arm_side_x is not None and glove_side_x is not None:
        plt.text(arm_side_x, -23, 'Arm Side', fontsize=12, verticalalignment='center', horizontalalignment='center',
                 bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))
        plt.text(glove_side_x, -23, 'Glove Side', fontsize=12, verticalalignment='center', horizontalalignment='center',
                 bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))
    
    # Label the axes and add the title
    plt.xlabel('Horizontal Break (inches)')
    plt.ylabel('Induced Vertical Break (inches)')
    plt.title(f'Pitch Breaks: Horizontal vs Vertical Break for {pitcher_name}', fontsize=16)
    
    # Adjust the legend and layout, then save the plot to the PDF
    plt.legend(title='Pitch Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    pdf.savefig()
    plt.close()

    

# Main function to create directories, save raw data, and generate PDFs
def generate_visuals_and_rawdata(df):
    # Prompt the user for the minimum date to generate individual reports
    user_input_date = input("Enter the minimum date (YYYY-MM-DD) to generate individual reports: ")
    min_date = pd.to_datetime(user_input_date)

    base_dir = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\pdf"
    for (pitcher, observation_date), pitcher_data in df.groupby(['Pitcher', 'Date']):
        try:
            # Extract and format pitcher names
            last_name, first_name = pitcher.split(", ")
            pitcher_formatted = f"{first_name} {last_name}"
            pitcher_folder = os.path.join(base_dir, f"{first_name}_{last_name}")

            # Ensure the pitcher folder exists
            safe_create_dir(pitcher_folder)

            # Parse the observation date properly and check against min_date
            observation_date_obj = pd.to_datetime(observation_date)
            formatted_date = observation_date_obj.strftime('%Y-%m-%d')

            # Create cumulative data and report
            cumulative_path = os.path.join(pitcher_folder, f"{first_name}_{last_name}_cumulative_rawdata.csv")
            if os.path.exists(cumulative_path):
                existing_data = pd.read_csv(cumulative_path)
                cumulative_data = pd.concat([existing_data, pitcher_data], ignore_index=True).drop_duplicates()
            else:
                cumulative_data = pitcher_data

            cumulative_data.to_csv(cumulative_path, index=False)

            # Save cumulative PDF report
            cumulative_pdf_path = os.path.join(pitcher_folder, f"{first_name}_{last_name}_cumulative_report.pdf")
            with PdfPages(cumulative_pdf_path) as cumulative_pdf:
                create_break_plot(cumulative_data, f"{pitcher_formatted} (Cumulative)", cumulative_pdf)
                plot_pitch_locations_by_pitcher(cumulative_data, f"{pitcher_formatted} (Cumulative)", cumulative_pdf)
                create_table_plot(cumulative_data, f"{pitcher_formatted} (Cumulative)", cumulative_pdf)
                create_release_plot(cumulative_data, f"{pitcher_formatted} (Cumulative)", cumulative_pdf)
                create_polar_plots(cumulative_data, f"{pitcher_formatted} (Cumulative)", cumulative_pdf)
                
                
                

            # Generate individual reports only for dates after the user input
            if observation_date_obj > min_date:
                print(f"{observation_date_obj} is greater than {min_date}")  # Log check
                date_folder = os.path.join(pitcher_folder, formatted_date)
                safe_create_dir(date_folder)

                # Save daily raw data
                daily_rawdata_path = os.path.join(date_folder, f"{first_name}_{last_name}_{formatted_date}_rawdata.csv")
                pitcher_data.to_csv(daily_rawdata_path, index=False)

                # Save daily PDF report
                pdf_report_path = os.path.join(date_folder, f"{first_name}_{last_name}_{formatted_date}_report.pdf")
                with PdfPages(pdf_report_path) as pdf:
                    create_break_plot(pitcher_data, pitcher_formatted, pdf)
                    plot_pitch_locations_by_pitcher(pitcher_data, pitcher_formatted, pdf)
                    create_polar_plots(pitcher_data, pitcher_formatted, pdf)
                    create_release_plot(pitcher_data, pitcher_formatted, pdf)
                    create_table_plot(pitcher_data, pitcher_formatted, pdf)
                    

            print(f"Saved data and reports for {pitcher_formatted} on {observation_date}")

        except Exception as e:
            print(f"Error processing {pitcher} on {observation_date}: {e}")

# Execute the function
generate_visuals_and_rawdata(df)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the CSV file into a pandas DataFrame
file_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)

# Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

# Define pitch categories based on initial pitch types
pitch_categories = {
    "Breaking Ball": ["Slider", "Curveball"],
    "Fastball": ["Fastball", "Four-Seam", "Sinker", "Cutter"],
    "Offspeed": ["ChangeUp", "Splitter"]
}

# Function to categorize pitch types into broader groups
def categorize_pitch_type(pitch_type):
    for category, pitches in pitch_categories.items():
        if pitch_type in pitches:
            return category
    return None

# Create a new column 'PitchCategory' to categorize pitches
df['Pitchcategory'] = df['Autopitchtype'].apply(categorize_pitch_type)

# Function to create heatmaps for each batter, pitch category, and batter side in a grid layout
def visualize_heatmaps_by_batter(df):
    # Set up figure grid
    batters = df['Batter'].unique()
    pitch_categories_list = ["Breaking Ball", "Fastball", "Offspeed"]
    batter_sides = ['Left', 'Right']

    # Create a figure with dynamic size based on the number of batters
    fig, axes = plt.subplots(len(batters) * 2, len(pitch_categories_list), figsize=(20, len(batters) * 10))
    
    # Loop through each batter, pitch category, and batter side
    for i, batter in enumerate(batters):
        for j, pitch_category in enumerate(pitch_categories_list):
            for k, batter_side in enumerate(batter_sides):
                # Select data for specific batter, pitch category, and batter side
                batter_data = df[(df['Batter'] == batter) & 
                                 (df['Pitchcategory'] == pitch_category)]
                
                # Check if there's sufficient variation in the data
                if batter_data.empty or batter_data['Exitspeed'].nunique() < 2:
                    continue
                
                # Determine which subplot to use: top row for 'Left', bottom row for 'Right'
                ax = axes[i * 2 + k, j] if len(batters) > 1 else axes[k, j]
                
                # Create the heatmap
                sns.kdeplot(
                    x=batter_data['Platelocside'],
                    y=batter_data['Platelocheight'],
                    weights=batter_data['Exitspeed'],
                    cmap="coolwarm",
                    fill=True,
                    ax=ax,
                    warn_singular=False
                )
                
                # Add the strike zone
                ax.add_patch(plt.Rectangle((-0.83, 1.5), 1.66, 2.1, edgecolor='black', facecolor='none'))
                
                # Add labels based on batter side
                if batter_side == 'Right':
                    ax.text(-2.2, 2, 'Stands Here', fontsize=10, rotation=90, verticalalignment='center',
                            bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))
                else:
                    ax.text(2.2, 2, 'Stands Here', fontsize=10, rotation=270, verticalalignment='center',
                            bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))

                # Customize subplot title
                ax.set_title(f'{batter} - {pitch_category}', fontsize=12)
                ax.set_xlim(-2.5, 2.5)
                ax.set_ylim(0, 5)
                ax.set_xlabel('Horizontal Plate Location')
                ax.set_ylabel('Vertical Plate Location')
    
    # Add padding between subplots
    plt.subplots_adjust(hspace=0.5, wspace=0.3)

    plt.tight_layout()
    plt.show()

# Call the function to visualize heatmaps
visualize_heatmaps_by_batter(df)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the CSV file into a pandas DataFrame
file_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)

# Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

# Define pitch categories based on initial pitch types
pitch_categories = {
    "Breaking Ball": ["Slider", "Curveball"],
    "Fastball": ["Fastball", "Four-Seam", "Sinker", "Cutter"],
    "Offspeed": ["ChangeUp", "Splitter"]
}

# Function to categorize pitch types into broader groups
def categorize_pitch_type(pitch_type):
    for category, pitches in pitch_categories.items():
        if pitch_type in pitches:
            return category
    return None

# Create a new column 'PitchCategory' to categorize pitches
df['Pitchcategory'] = df['Autopitchtype'].apply(categorize_pitch_type)

# Set up the color palette based on pitch type
pitch_types = df['Autopitchtype'].unique()
palette = sns.color_palette('hsv', len(pitch_types))
color_map = dict(zip(pitch_types, palette))

# Function to create scatter plots for each pitcher with pitch locations, separated by batter side
def plot_pitch_locations_by_pitcher(df):
    pitchers = df['Pitcher'].unique()

    # Loop through each pitcher
    for pitcher in pitchers:
        pitcher_data = df[df['Pitcher'] == pitcher]

        # Create a figure with two subplots: one for Right, one for Left
        fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
        batter_sides = ['Right', 'Left']

        for i, batter_side in enumerate(batter_sides):
            side_data = pitcher_data[pitcher_data['Batterside'] == batter_side]

            # Create scatter plot for pitch locations, colored by pitch type
            sns.scatterplot(
                data=side_data,
                x='Platelocside',
                y='Platelocheight',
                hue='Autopitchtype',
                palette=color_map,
                s=100,
                edgecolor='black',
                ax=axes[i]
            )

            # Add the strike zone
            axes[i].add_patch(plt.Rectangle((-0.83, 1.5), 1.66, 2.1, edgecolor='black', facecolor='none'))

            # Customize the plot
            axes[i].set_title(f'{pitcher} vs {batter_side} Handed Batters', fontsize=14)
            axes[i].set_xlim(-2.5, 2.5)
            axes[i].set_ylim(0, 5)
            axes[i].set_xlabel('Horizontal Plate Location')
            axes[i].set_ylabel('Vertical Plate Location')
            axes[i].legend(title='Pitch Type', bbox_to_anchor=(1.05, 1), loc='upper left')

        plt.tight_layout()
        plt.show()

# Call the function to plot pitch locations by pitcher
plot_pitch_locations_by_pitcher(df)


In [ ]:
>>> from pybaseball import statcast 

pd.set_option('display.max_columns', None)

>>> data = statcast(start_dt='2017-06-24', end_dt='2017-06-27')
>>> data.head(2)